# DATA CLEANING PART

### Column removal

In [ ]:
import pandas as pd
import os

def process_csv(source_file, target_dir, output_name, columns_to_drop):
    # Load
    df = pd.read_csv(source_file)
    
    # Drop columns
    df.drop(columns=columns_to_drop, inplace=True)
    
    # Create target directory if needed
    os.makedirs(target_dir, exist_ok=True)
    
    # Save
    output_path = os.path.join(target_dir, output_name)
    df.to_csv(output_path, index=False)
    
    return output_path

#process_csv("artist_reactions.csv", "processed/", "artist_reactions.csv", ["id", "reacted_at", "sentiment", "on_user_id"])

#process_csv("artists_1.csv", "processed/", "artists_1.csv", ["description", "score", "rank", "created_at", "updated_at",
                                                            # "likes_count", "dislikes_count", "reactions_count", "cover_id"])

#process_csv("covers.csv", "processed/", "covers.csv", ["file_id", "unique_file_id", "file_format", "mime_type", "file_size", "file_url", "width", "height",
                                                       #"uploaded_by", "source", "updated_at", "created_at"])

#process_csv("reaction_types.csv", "processed/", "reaction_types.csv", ["emoji", "sentiment", "description"])

#process_csv("track_reactions.csv", "processed/", "track_reactions.csv", ["reacted_at", "genre_id", "id", "sentiment", "on_user_id"])

#process_csv("tracks.csv", "processed/", "tracks.csv", ["file_id", "unique_file_id", "file_type", "mime_type", "extension", "title", 
                                           #            "duration", "album_id", "chat_id", "score", "rank", "created_at", "updated_at",
                                             #          "likes_count", "dislikes_count", "reactions_count", "cover_id", "metadata", "performer", "message_id"])


'processed/tracks.csv'

In [15]:
import pandas as pd

def remove_brackets(filepath, columns, save=False):
    """
    Remove exactly one leading '{' and one trailing '}' from each value in specified columns.
    
    Parameters:
    filepath : str - path to the CSV file.
    columns : list - column names to clean.
    save : bool (default False) - if True, overwrite the original file.
    
    Returns:
    pandas.DataFrame - cleaned DataFrame.
    """
    df = pd.read_csv(filepath)
    
    for col in columns:
        if col not in df.columns:
            print(f"Warning: Column '{col}' not found – skipping.")
            continue
        
        # Apply the transformation to each cell
        df[col] = df[col].astype(str).apply(
            lambda s: s[1:-1] if s.startswith('{') and s.endswith('}') else s
        )
    
    if save:
        df.to_csv(filepath, index=False)
        print(f"Changes saved to {filepath}")
    
    return df

cleaned = remove_brackets("processed/tracks.csv", ["artists_id", "uploaded_by"], True)

Changes saved to processed/tracks.csv


In [16]:
import pandas as pd

# File paths
FIXED_FILE = 'fixed_flagged.csv'
FLAGGED_FILE = 'flagged_tracks.csv'
TRACKS_FILE = 'processed/tracks.csv'          # cleaned tracks
ARTISTS_FILE = 'processed/artists_1.csv'
LOG_FILE = 'artist_fill_log.txt'

# Load data
fixed = pd.read_csv(FIXED_FILE)              # id, title, performer
flagged = pd.read_csv(FLAGGED_FILE)          # id, title, performer, anomaly_reason
tracks = pd.read_csv(TRACKS_FILE)
artists = pd.read_csv(ARTISTS_FILE)

# Filter tracks that have "Missing Data" in the anomaly_reason column
missing_data_mask = flagged['anomaly_reason'].str.contains('Missing Data', case=False, na=False)
flagged_missing = flagged[missing_data_mask]
target_ids = set(flagged_missing['id'])

# Keep only rows in fixed_flagged that belong to those target IDs
fixed_to_process = fixed[fixed['id'].isin(target_ids)]

# Build artist name -> id lookup (lowercase)
artist_name_to_id = {}
for _, row in artists.iterrows():
    if pd.notna(row['name']):
        key = str(row['name']).strip().lower()
        if key not in artist_name_to_id:      # keep first occurrence to avoid duplicate keys
            artist_name_to_id[key] = row['id']

# Process
success_count = 0
fail_count = 0
log_lines = []

for _, frow in fixed_to_process.iterrows():
    track_id = frow['id']
    performer = str(frow['performer']).strip() if pd.notna(frow['performer']) else ''

    # Find the track in the tracks DataFrame
    track_idx = tracks.index[tracks['id'] == track_id]
    if track_idx.empty:
        msg = f"Track {track_id} not found in tracks.csv – skipping."
        log_lines.append(msg)
        fail_count += 1
        continue
    track_idx = track_idx[0]

    # Check current artists_id
    current_artist = tracks.at[track_idx, 'artists_id']
    if pd.notna(current_artist) and str(current_artist).strip() != '':
        msg = f"Track {track_id} already has artist(s) {current_artist} – skipping."
        log_lines.append(msg)
        continue

    # If performer is empty, we can't fill anything
    if not performer:
        msg = f"Track {track_id}: performer is empty – skipping."
        log_lines.append(msg)
        fail_count += 1
        continue

    # Look up performer in artists_1
    performer_key = performer.lower().strip()
    if performer_key in artist_name_to_id:
        artist_id = artist_name_to_id[performer_key]
        tracks.at[track_idx, 'artists_id'] = artist_id
        msg = f"Track {track_id}: set artist to {artist_id} ('{performer}')"
        log_lines.append(msg)
        success_count += 1
    else:
        msg = f"Track {track_id}: performer '{performer}' not found in artists_1 – skipping."
        log_lines.append(msg)
        fail_count += 1

# Save updated tracks
tracks.to_csv(TRACKS_FILE, index=False)

# Write log
with open(LOG_FILE, 'w', encoding='utf-8') as f:
    f.write(f"Processed {len(fixed_to_process)} tracks flagged with 'Missing Data'.\n")
    f.write(f"Successes: {success_count}\nFailures: {fail_count}\n\n")
    f.write('\n'.join(log_lines))

# Report
print(f"Tracks processed: {len(fixed_to_process)}")
print(f"Successes: {success_count}")
print(f"Failures: {fail_count}")
print(f"Details written to {LOG_FILE}")

Tracks processed: 375
Successes: 176
Failures: 199
Details written to artist_fill_log.txt


In [18]:
import pandas as pd
import re
from thefuzz import fuzz

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
LOG_INPUT = 'artist_fill_log.txt'          # log from the previous script
TRACKS_FILE = 'processed/tracks.csv'
ARTISTS_FILE = 'processed/artists_1.csv'
LOG_OUTPUT = 'fuzzy_fill_log.txt'

# Fuzzy matching thresholds
MIN_SCORE = 95             # minimum token_sort_ratio to consider a match
MIN_GAP = 10               # required gap between best and second‑best score

# ------------------------------------------------------------
# 1. Parse log for "not found in artists_1" entries
# ------------------------------------------------------------
pattern = r"Track (\d+): performer '(.+?)' not found in artists_1"
cases = []

with open(LOG_INPUT, 'r', encoding='utf-8') as f:
    for line in f:
        match = re.search(pattern, line)
        if match:
            track_id = int(match.group(1))
            performer = match.group(2)
            cases.append((track_id, performer))

print(f"Found {len(cases)} 'not found' entries in log.")

# ------------------------------------------------------------
# 2. Load artists and build lookup structures
# ------------------------------------------------------------
artists = pd.read_csv(ARTISTS_FILE)

# List of (artist_id, original_name) for fuzzy matching
artist_info = []
for _, row in artists.iterrows():
    if pd.notna(row['name']):
        artist_info.append((row['id'], str(row['name']).strip()))

print(f"Loaded {len(artist_info)} artists.")

# ------------------------------------------------------------
# 3. Load tracks
# ------------------------------------------------------------
tracks = pd.read_csv(TRACKS_FILE)

# ------------------------------------------------------------
# 4. Fuzzy matching and updating
# ------------------------------------------------------------
success = 0
fail = 0
log_entries = []

for track_id, performer in cases:
    # Find track index
    track_idx = tracks.index[tracks['id'] == track_id]
    if track_idx.empty:
        log_entries.append(f"Track {track_id} not found in tracks.csv – skipping.")
        fail += 1
        continue
    track_idx = track_idx[0]

    # Ensure artists_id is still empty
    current = tracks.at[track_idx, 'artists_id']
    if pd.notna(current) and str(current).strip() != '':
        log_entries.append(f"Track {track_id} already has artist(s) – skipping.")
        continue

    # Normalise performer for matching
    perf_clean = performer.lower().strip()
    # (We keep the original for logging)

    # Compute token_sort_ratio against every artist name
    scores = []
    for aid, name in artist_info:
        name_clean = name.lower().strip()
        score = fuzz.token_sort_ratio(perf_clean, name_clean)
        scores.append((score, aid, name))

    # Sort descending by score
    scores.sort(key=lambda x: x[0], reverse=True)

    best_score, best_id, best_name = scores[0]
    second_score = scores[1][0] if len(scores) > 1 else 0

    if best_score >= MIN_SCORE and (best_score - second_score) >= MIN_GAP:
        # Accept match
        tracks.at[track_idx, 'artists_id'] = best_id
        log_entries.append(
            f"Track {track_id}: fuzzy matched '{performer}' -> {best_id} "
            f"('{best_name}', score={best_score}, gap={best_score - second_score})"
        )
        success += 1
    else:
        log_entries.append(
            f"Track {track_id}: no confident match for '{performer}' "
            f"(best='{best_name}' score={best_score}, gap={best_score - second_score})"
        )
        fail += 1

# ------------------------------------------------------------
# 5. Save updated tracks and write log
# ------------------------------------------------------------
tracks.to_csv(TRACKS_FILE, index=False)

with open(LOG_OUTPUT, 'w', encoding='utf-8') as f:
    f.write(f"Fuzzy matching results.\n")
    f.write(f"Successes: {success}\nFailures: {fail}\n\n")
    f.write('\n'.join(log_entries))

print(f"Fuzzy matching complete – {success} updated, {fail} skipped.")
print(f"Details written to {LOG_OUTPUT}")

Found 80 'not found' entries in log.
Loaded 5234 artists.
Fuzzy matching complete – 4 updated, 76 skipped.
Details written to fuzzy_fill_log.txt


In [20]:
import pandas as pd
import json
import unicodedata
import re
from collections import defaultdict

# ---------------------------------------------------------------------------
# 1. Artist name normalization function
# ---------------------------------------------------------------------------
INVISIBLE_CODES = {
    0x200B, 0x200C, 0x200D, 0xFEFF, 0x2060,  # zero‑width / BOM
    0x3164,                                    # Hangul filler
}

def normalize_artist_name(name: str) -> str:
    if pd.isna(name):
        return ""
    # NFKD normalization
    name = unicodedata.normalize('NFKD', name)
    # Remove diacritics (combining marks)
    name = ''.join(c for c in name if not unicodedata.combining(c))
    # Remove invisible formatting characters
    name = ''.join(c for c in name if ord(c) not in INVISIBLE_CODES
                   and unicodedata.category(c) != 'Cf')
    # Lowercase
    name = name.lower()
    # Collapse whitespace
    name = ' '.join(name.split())
    # Strip leading/trailing punctuation
    name = name.strip('.,-;:!?\'" ')
    return name

# ---------------------------------------------------------------------------
# 2. Load data
# ---------------------------------------------------------------------------
print("Loading data...")
artists = pd.read_csv('processed/artists_1.csv')       # id, name, metadata
tracks = pd.read_csv('processed/tracks.csv')           # id, artists_id, ...

# Quick stats before dedup
original_artist_count = len(artists)
tracks_missing_artist = tracks['artists_id'].isna().sum() + (tracks['artists_id'].astype(str).str.strip() == '').sum()
print(f"Artists table: {original_artist_count} rows")
print(f"Tracks with missing artist_id: {tracks_missing_artist}")

# ---------------------------------------------------------------------------
# 3. Normalize and deduplicate artists
# ---------------------------------------------------------------------------
print("\nNormalizing artist names...")
artists['normalized_name'] = artists['name'].apply(normalize_artist_name)

# Group by normalized name, pick smallest ID as canonical
name_to_canonical = {}
id_to_canonical = {}          # original_id -> canonical_id
groups = defaultdict(list)

for _, row in artists.iterrows():
    norm = row['normalized_name']
    if norm == "":
        # Artists with empty names keep their own ID (can't group)
        canonical_id = row['id']
    else:
        groups[norm].append(row['id'])

# Assign canonical IDs
for norm, ids in groups.items():
    canonical_id = min(ids)
    name_to_canonical[norm] = canonical_id
    for id_ in ids:
        id_to_canonical[id_] = canonical_id

# Handle empty-named artists individually
for _, row in artists.iterrows():
    if row['normalized_name'] == "":
        id_to_canonical[row['id']] = row['id']

canonical_artists = set(id_to_canonical.values())
duplicate_groups = {norm: ids for norm, ids in groups.items() if len(ids) > 1}

print(f"Canonical artists after dedup: {len(canonical_artists)}")
print(f"Duplicate groups (merged): {len(duplicate_groups)}")
merged_artist_ids = sum(len(ids) - 1 for ids in duplicate_groups.values())
print(f"Artist IDs merged into others: {merged_artist_ids}")

# Save id mapping
mapping_df = pd.DataFrame(list(id_to_canonical.items()),
                          columns=['original_id', 'canonical_id'])
mapping_df.to_csv('artist_dedup_mapping.csv', index=False)
print("Saved artist_dedup_mapping.csv")

# ---------------------------------------------------------------------------
# 4. Parse metadata and extract genres + related_artists
# ---------------------------------------------------------------------------
print("\nParsing metadata...")

def parse_metadata(meta_str):
    """Return (genres_list, related_artists_list) or (None, None) on failure."""
    if pd.isna(meta_str) or meta_str.strip() in ('', '{}'):
        return None, None
    try:
        # Fix double-quoting typical in CSV
        cleaned = meta_str.strip()
        if cleaned.startswith('"') and cleaned.endswith('"'):
            cleaned = cleaned[1:-1]
        cleaned = cleaned.replace('""', '"')
        data = json.loads(cleaned)
        genres = data.get('genres', [])
        related = data.get('related_artists', [])
        return genres, related
    except Exception:
        return None, None

# We'll only process canonical artists (one row per canonical ID, using the smallest ID's metadata)
canonical_info = {}
for canonical_id in canonical_artists:
    # Get the row of the canonical ID itself (it exists)
    row = artists[artists['id'] == canonical_id].iloc[0]
    genres, related = parse_metadata(row['metadata'])
    canonical_info[canonical_id] = {
        'name': row['name'],
        'normalized_name': row['normalized_name'],
        'genres': genres,
        'related_artists': related
    }

# Build a lookup from normalized name to canonical ID for matching
norm_to_canonical = {}
for canon_id, info in canonical_info.items():
    norm_name = info['normalized_name']
    if norm_name:
        norm_to_canonical[norm_name] = canon_id

# ---------------------------------------------------------------------------
# 5. Match related_artists to canonical IDs
# ---------------------------------------------------------------------------
print("Matching related artists...")
links = []                 # (canonical_artist_id, related_canonical_id)
unmatched = []
artists_with_related = 0
total_related_entries = 0

for canon_id, info in canonical_info.items():
    related_list = info['related_artists']
    if not related_list:
        continue
    artists_with_related += 1
    total_related_entries += len(related_list)
    for rel_name in related_list:
        norm_rel = normalize_artist_name(rel_name)
        if norm_rel and norm_rel in norm_to_canonical:
            matched_id = norm_to_canonical[norm_rel]
            links.append((canon_id, matched_id))
        else:
            unmatched.append((canon_id, info['name'], rel_name))

# Remove self-links and duplicates
links = list(set(links))
links = [(a, b) for a, b in links if a != b]

matched_edges = len(links)
print(f"Artists with related_artists data: {artists_with_related}")
print(f"Total related_artist entries: {total_related_entries}")
print(f"Matched edges (after dedup & self-removal): {matched_edges}")
print(f"Unmatched related names: {len(unmatched)}")

# Save links
links_df = pd.DataFrame(links, columns=['artist_id', 'related_artist_id'])
links_df.to_csv('artist_links.csv', index=False)
print("Saved artist_links.csv")

# ---------------------------------------------------------------------------
# 6. Genre statistics and output
# ---------------------------------------------------------------------------
print("\nGenre analysis...")
all_genres = set()
genre_rows = []          # (artist_id, genre)
artists_with_genres = 0

for canon_id, info in canonical_info.items():
    genres = info['genres']
    if genres:
        artists_with_genres += 1
        for g in genres:
            all_genres.add(g)
            genre_rows.append((canon_id, g))

print(f"Artists with genre information: {artists_with_genres}")
print(f"Total unique genres: {len(all_genres)}")
if all_genres:
    print(f"Sample genres: {list(sorted(all_genres))[:30]}")

genre_df = pd.DataFrame(genre_rows, columns=['artist_id', 'genre'])
genre_df.to_csv('artist_genres.csv', index=False)
print("Saved artist_genres.csv")

# ---------------------------------------------------------------------------
# 7. Summary statistics
# ---------------------------------------------------------------------------
print("\n========== SUMMARY ==========")
print(f"Original artist count: {original_artist_count}")
print(f"After dedup: {len(canonical_artists)}")
print(f"Merged IDs: {merged_artist_ids}")
print(f"Artists with metadata (genres): {artists_with_genres}")
print(f"Unique genres: {len(all_genres)}")
print(f"Artists with related_artists: {artists_with_related}")
print(f"Total related entries: {total_related_entries}")
print(f"Successful edges: {matched_edges}")
print(f"Tracks missing artist (original): {tracks_missing_artist}")

Loading data...
Artists table: 5234 rows
Tracks with missing artist_id: 284

Normalizing artist names...
Canonical artists after dedup: 5152
Duplicate groups (merged): 79
Artist IDs merged into others: 82
Saved artist_dedup_mapping.csv

Parsing metadata...
Matching related artists...
Artists with related_artists data: 4233
Total related_artist entries: 20997
Matched edges (after dedup & self-removal): 6857
Unmatched related names: 14134
Saved artist_links.csv

Genre analysis...
Artists with genre information: 3426
Total unique genres: 1997
Sample genres: ['#acoustic #Guitar #instrumental #Cover', '#iran_music', '#late_night_material', '#look_at_it', "'77 punk", '.', '11', '13', '14', '1960s', '2-step', '2000s', '2003', '20th Century', '21st Century Vampire', '2afm', '3', '3 Doors Down', '40s', '4ad', '5 Stars', '50 Cent', '50s', '60s', "70's", '70s', '8-bit', '80ies', '80s', '8bit']
Saved artist_genres.csv

========== SUMMARY ==========
Original artist count: 5234
After dedup: 5152
Mer

In [21]:
import pandas as pd
import unicodedata
import re
from collections import Counter
from thefuzz import fuzz

# ---------------------------------------------------------------------------
# 1. Genre normalisation
# ---------------------------------------------------------------------------

# Words that are already singular but end with 's' – we don't strip the 's' from them.
PROTECTED_WORDS = {
    "blues", "classics", "soul", "jazz", "funk", "rock", "pop", "folk",
    "indie", "metal", "punk", "rap", "hip", "hop", "trip", "electronic",
    "dance", "latin", "classical", "country", "reggae", "disco", "techno",
    "house", "ambient", "industrial", "gospel", "opera", "orchestral",
    "soundtrack", "experimental", "alternative", "progressive", "garage",
    "synth", "new", "wave", "post", "grunge", "hardcore", "thrash",
    "doom", "shoegaze", "dream", "art", "electro", "dubstep", "drum",
    "bass", "trance", "hardstyle", "downtempo", "chillout", "lounge",
    "new", "age", "world", "fusion", "bossa", "nova", "samba", "tango",
    "swing", "bluegrass", "americana", "zydeco", "cajun", "emo",
    "screamo", "math", "noise", "drone", "avant", "garde", "minimalism",
    "baroque", "romantic", "renaissance", "medieval", "gregorian",
    "choral", "choir", "a", "cappella", "male", "female", "singer",
    "songwriter", "composer", "conductor", "rapper", "mc", "dj",
    "producer", "band", "orchestra", "ensemble", "duo", "trio",
    "quartet", "quintet", "sextet", "septet", "octet", "nonet", "big",
    "brass", "marching", "military", "pipe", "corps", "gospel",
    "opera", "chorus", "ballet", "dance", "modern", "contemporary",
    "improvisation", "free", "jazz", "modal", "bebop", "cool",
    "hard", "bop", "post", "fusion", "latin", "jazz", "smooth",
    "acid", "jazz", "nu", "jazz", "electro", "swing", "gypsy",
    "jazz", "ragtime", "boogie", "woogie", "stride", "honky",
    "tonk", "western", "swing", "rockabilly", "psychobilly", "punk",
    "anarcho", "hardcore", "post", "metalcore", "deathcore",
    "grindcore", "noisecore", "sludge", "stoner", "doom", "drone",
    "psychedelic", "space", "krautrock", "progressive", "art",
    "glam", "rock", "new", "romantic", "synthpop", "electropop",
    "dance", "pop", "europop", "j", "pop", "k", "pop", "c", "pop",
    "v", "pop", "t", "pop", "arab", "pop", "persian", "pop",
    "turkish", "pop", "latin", "pop", "reggaeton", "latin", "trap",
    "dembow", "baile", "funk", "axé", "sertanejo", "forró",
    "pagode", "choro", "maracatu", "frevo", "samba", "bossa",
    "nova", "tropicalia", "mpb", "brazilian", "brazilian", "music",
    "latin", "alternative", "latin", "rock", "rock", "en", "español",
    "nueva", "canción", "trova", "son", "salsa", "merengue",
    "bachata", "cumbia", "vallenato", "ranchera", "mariachi",
    "norteño", "tejano", "conjunto", "banda", "duranguense",
    "pasito", "duranguense", "grupera", "romántica", "balada",
    "bolero", "tango", "milonga", "chacarera", "zamba", "cuarteto",
    "rock", "nacional", "rock", "argentino", "rock", "chileno",
    "rock", "mexicano", "rock", "peruano", "rock", "colombiano",
    "rock", "venezolano", "rock", "uruguayo", "rock", "brasileiro",
    "rock", "en", "tu", "idioma", "ska", "reggae", "dancehall",
    "ragga", "dub", "roots", "reggae", "lovers", "rock", "rocksteady",
    "ska", "punk", "reggae", "fusion", "afrobeat", "afrobeats",
    "highlife", "juju", "fuji", "apala", "makossa", "soukous",
    "kwaito", "kizomba", "zouk", "coupe", "decalé", "ndombolo",
    "bikutsi", "mbalax", "taarab", "benga", "ohangla", "genge",
    "kapuka", "mugithi", "bongo", "flava", "singeli", "taureg",
    "desert", "blues", "rai", "chaabi", "gnawa", "andalusi",
    "malouf", "tarab", "maqam", "dastgah", "radif", "qawwali",
    "ghazal", "sufi", "bhangra", "filmi", "indian", "classical",
    "hindustani", "carnatic", "thumri", "dhrupad", "khyal",
    "tappa", "tarana", "bhajan", "kirtan", "shabad", "santoor",
    "sitar", "sarod", "veena", "tabla", "mridangam", "ghatam",
    "kanjira", "morsing", "flute", "shehnai", "nadaswaram",
    "violin", "sarangi", "dilruba", "esraj", "harmonium",
    "surbahar", "rudra", "veena", "vichitra", "veena", "chitra",
    "veena", "gotuvadyam", "jal", "tarang", "kasht", "tarang",
    "manjira", "tambura", "tanpura", "swarmandal", "santur",
    "qanun", "oud", "ney", "kamancheh", "duduk", "zurna", "saz",
    "baglama", "bouzouki", "tambouras", "tzouras", "mandolin",
    "balalaika", "domra", "gusli", "bandura", "kobza", "torban",
    "cimbalom", "hammered", "dulcimer", "psaltery", "harpsichord",
    "clavichord", "virginal", "spinet", "organ", "positive",
    "portative", "regal", "harmonium", "accordion", "concertina",
    "bandoneon", "melodeon", "piano", "fortepiano", "square",
    "upright", "grand", "prepared", "tack", "electric", "digital",
    "synthesizer", "moog", "arp", "sequential", "circuits",
    "roland", "korg", "yamaha", "casio", "nord", "kurzweil",
    "waldorf", "access", "novation", "elektron", "dave", "smith",
    "oberheim", "prophet", "jupiter", "juno", "sh", "101", "tb",
    "303", "tr", "808", "tr", "909", "mpc", "sampler", "groovebox",
    "drum", "machine", "rhythm", "beatbox", "loop", "sequencer",
    "arpeggiator", "modular", "west", "coast", "east", "coast",
    "experimental", "noise", "ambient", "drone", "field", "recording",
    "musique", "concrete", "electroacoustic", "acousmatic",
    "spectral", "microtonal", "just", "intonation", "temperament",
    "xenharmonic", "algorithmic", "generative", "live", "coding",
    "circuit", "bending", "glitch", "chiptune", "8", "bit",
    "bitpop", "videogame", "music", "chipmusic", "demoscene",
    "tracker", "mod", "xm", "it", "s3m", "midi", "general",
    "midi", "soundfont", "vst", "au", "rtas", "aax", "plugin",
    "daw", "ableton", "logic", "cubase", "pro", "tools", "fl",
    "studio", "reason", "bitwig", "studio", "one", "reaper",
    "garageband", "audacity", "sound", "forge", "wavelab", "ozone",
    "izotope", "waves", "native", "instruments", "arturia",
    "spitfire", "orchestral", "tools", "eastwest", "vienna",
    "symphonic", "library", "cinematic", "epic", "trailer",
    "soundscape", "atmosphere", "texture", "pad", "drone",
    "minimal", "maximal", "complex", "simple", "melodic",
    "rhythmic", "harmonic", "dissonant", "consonant", "tonal",
    "atonal", "modal", "chromatic", "pentatonic", "whole", "tone",
    "octatonic", "serial", "twelve", "tone", "aleatoric",
    "indeterminate", "graphic", "score", "chance", "happening",
    "fluxus", "sound", "art", "installation", "performance",
    "theatre", "opera", "musical", "cabaret", "vaudeville",
    "burlesque", "revue", "variety", "circus", "carnival",
    "masquerade", "parade", "procession", "pageant", "spectacle",
    "extravaganza", "show", "concert", "recital", "jam", "session",
    "improv", "open", "mic", "battle", "competition", "festival",
    "rave", "party", "club", "disco", "ball", "prom", "wedding",
    "funeral", "ceremony", "ritual", "rite", "mass", "liturgy",
    "hymn", "chorale", "anthem", "song", "aria", "recitative",
    "chorus", "duet", "trio", "quartet", "quintet", "sextet",
    "septet", "octet", "nonet", "ensemble", "orchestra", "band",
    "group", "collective", "project", "alias", "pseudonym",
    "moniker", "stage", "name", "persona", "character", "role",
    "actor", "actress", "performer", "artist", "musician", "singer",
    "rapper", "poet", "writer", "composer", "producer", "engineer",
    "mixer", "mastering", "recording", "studio", "live", "bootleg",
    "unofficial", "official", "release", "album", "ep", "single",
    "compilation", "soundtrack", "score", "mixtape", "demo",
    "promo", "sampler", "anthology", "box", "set", "collection",
    "greatest", "hits", "best", "of", "essential", "introductory",
    "retrospective", "tribute", "cover", "remix", "rework", "edit",
    "version", "take", "cut", "track", "medley", "megamix",
    "mashup", "blend", "transition", "segue", "interlude",
    "prelude", "postlude", "overture", "finale", "coda",
    "introduction", "outro", "fade", "crossfade", "overdub",
    "multitrack", "stereo", "mono", "surround", "quadraphonic",
    "binaural", "dolby", "dts", "thx", "hi", "fi", "lo", "fi",
    "high", "fidelity", "low", "fidelity", "analog", "digital",
    "warm", "cold", "bright", "dark", "clear", "muddy", "punchy",
    "smooth", "rough", "raw", "polished", "produced", "unproduced",
    "arranged", "orchestrated", "composed", "improvised",
    "spontaneous", "structured", "free", "tight", "loose", "groove",
    "swing", "shuffle", "straight", "syncopated", "offbeat",
    "backbeat", "downbeat", "upbeat", "tempo", "bpm", "fast",
    "slow", "medium", "moderate", "largo", "adagio", "andante",
    "moderato", "allegro", "presto", "vivace", "accelerando",
    "ritardando", "rubato", "dynamics", "piano", "forte",
    "crescendo", "decrescendo", "sforzando", "legato", "staccato",
    "portamento", "glissando", "tremolo", "vibrato", "trill",
    "grace", "note", "appoggiatura", "acciaccatura", "mordent",
    "turn", "arpeggio", "chord", "cluster", "harmony", "melody",
    "counterpoint", "polyphony", "homophony", "heterophony",
    "monophony", "drone", "ostinato", "riff", "lick", "motif",
    "theme", "subject", "variation", "development", "exposition",
    "recapitulation", "coda", "sonata", "form", "binary", "ternary",
    "rondo", "theme", "and", "variations", "fugue", "canon",
    "invention", "suite", "partita", "toccata", "fantasia",
    "prelude", "postlude", "impromptu", "ballade", "scherzo",
    "minuet", "gavotte", "sarabande", "gigue", "bourrée",
    "rigaudon", "passepied", "loure", "polonaise", "mazurka",
    "waltz", "ländler", "galop", "polka", "schottische",
    "ecossaise", "contredanse", "quadrille", "cotillion",
    "hornpipe", "reel", "jig", "strathspey", "airs", "lament",
    "march", "fanfare", "processional", "recessional", "national",
    "anthem", "folk", "song", "ballad", "epic", "legend", "myth",
    "fable", "tale", "story", "narrative", "lyric", "poem",
    "verse", "chorus", "refrain", "bridge", "middle", "eight",
    "intro", "outro", "hook", "drop", "build", "breakdown",
    "climax", "resolution", "denouement", "ending", "fade", "out",
    "cold", "end", "loop", "repeat", "coda", "sign", "dal",
    "segno", "da", "capo", "fine", "volta", "prima", "volta",
    "seconda", "volta", "terza", "volta", "quarta", "volta",
    "quinta", "volta", "sesta", "volta", "settima", "volta",
    "ottava", "volta", "nona", "volta", "decima", "volta",
    "undicesima", "volta", "dodicesima", "volta", "tredicesima",
    "volta", "quattordicesima", "volta", "quindicesima", "volta",
    "sedicesima", "volta", "diciassettesima", "volta",
    "diciannovesima", "volta", "ventesima", "volta"
}


def normalize_genre(genre: str) -> str:
    """Normalize a single genre string to a canonical form."""
    # Lowercase and strip
    s = genre.strip().lower()
    # Replace hyphens, underscores, slashes with space
    s = re.sub(r'[–_/]', ' ', s)
    # Remove other punctuation (keep letters, digits, spaces)
    s = re.sub(r'[^\w\s]', '', s)
    # Collapse whitespace
    s = ' '.join(s.split())
    # Handle plurals: for each word, if it ends with 's' and is not in protected set, remove trailing 's'
    words = s.split()
    new_words = []
    for w in words:
        if w.endswith('s') and w not in PROTECTED_WORDS:
            w = w[:-1]
        new_words.append(w)
    return ' '.join(new_words)


def cluster_normalized(names, threshold=90):
    """
    Group normalized genre strings by fuzzy token sort ratio.
    Returns a list of clusters (each a list of names).
    """
    names = list(set(names))
    clusters = []
    used = set()
    for i, name in enumerate(names):
        if name in used:
            continue
        cluster = [name]
        used.add(name)
        for j in range(i + 1, len(names)):
            if names[j] in used:
                continue
            ratio = fuzz.token_sort_ratio(name, names[j])
            if ratio >= threshold:
                # Additional safety: require at least one common word
                set1 = set(name.split())
                set2 = set(names[j].split())
                if set1 & set2:
                    cluster.append(names[j])
                    used.add(names[j])
        clusters.append(cluster)
    return clusters


# ---------------------------------------------------------------------------
# 2. Load raw genres
# ---------------------------------------------------------------------------
print("Loading artist_genres.csv...")
genre_df = pd.read_csv('artist_genres.csv')   # columns: artist_id, genre
raw_genres = genre_df['genre'].dropna().unique()
print(f"Unique raw genres: {len(raw_genres)}")

# ---------------------------------------------------------------------------
# 3. Normalize and count frequencies
# ---------------------------------------------------------------------------
raw_to_normalized = {}
normalized_counter = Counter()
for g in raw_genres:
    norm = normalize_genre(g)
    raw_to_normalized[g] = norm
    normalized_counter[norm] += genre_df[genre_df['genre'] == g].shape[0]  # count total occurrences (artist-genre pairs)

# Alternatively, count just number of artists per raw genre? We'll use frequency for canonical selection later.
# Build a frequency dict for normalized forms (how many raw rows map to it)
norm_freq = Counter()
for g in raw_genres:
    norm_freq[raw_to_normalized[g]] += genre_df[genre_df['genre'] == g].shape[0]

# ---------------------------------------------------------------------------
# 4. First round: exact match merging (same normalized form)
# For each normalized form, pick canonical as the most frequent raw genre that maps to it.
# ---------------------------------------------------------------------------
normalized_to_raws = defaultdict(set)
for g in raw_genres:
    normalized_to_raws[raw_to_normalized[g]].add(g)

canonical_from_exact = {}  # normalized -> canonical (a raw genre string)
for norm, raws in normalized_to_raws.items():
    # Choose the raw with highest total artist count (popularity)
    best_raw = max(raws, key=lambda r: genre_df[genre_df['genre'] == r].shape[0])
    canonical_from_exact[norm] = best_raw

# ---------------------------------------------------------------------------
# 5. Fuzzy clustering of normalized forms
# ---------------------------------------------------------------------------
print("Clustering normalized genres...")
all_norms = list(normalized_to_raws.keys())
clusters = cluster_normalized(all_norms, threshold=90)
print(f"Number of fuzzy clusters: {len(clusters)}")

# For each cluster, pick a canonical label (most frequent raw across all normalized in the cluster)
cluster_canonical = {}
for cluster in clusters:
    if len(cluster) == 1:
        # already handled by exact mapping
        continue
    # Merge: choose the raw genre with highest overall count among all normalized forms in cluster
    best_raw = None
    best_count = -1
    for norm in cluster:
        raws = normalized_to_raws[norm]
        for r in raws:
            cnt = genre_df[genre_df['genre'] == r].shape[0]
            if cnt > best_count:
                best_count = cnt
                best_raw = r
    # Assign this canonical to all normalized in cluster
    for norm in cluster:
        canonical_from_exact[norm] = best_raw

# ---------------------------------------------------------------------------
# 6. Build final mapping raw -> canonical
# ---------------------------------------------------------------------------
raw_to_canonical = {}
for raw in raw_genres:
    norm = raw_to_normalized[raw]
    canon = canonical_from_exact.get(norm, raw)  # fallback to raw itself
    raw_to_canonical[raw] = canon

# Count merges (where raw != canonical)
merges = {raw: canon for raw, canon in raw_to_canonical.items() if raw != canon}
print(f"Total merges (raw changed): {len(merges)}")

# Save mapping
mapping_df = pd.DataFrame(list(raw_to_canonical.items()), columns=['raw_genre', 'canonical_genre'])
mapping_df.to_csv('genre_clean_mapping.csv', index=False)
print("Saved genre_clean_mapping.csv")

# Log only successful merges (changes) to file
log_lines = []
for raw, canon in merges.items():
    log_lines.append(f"'{raw}' -> '{canon}'")

with open('genre_merge_log.txt', 'w', encoding='utf-8') as f:
    f.write(f"Total merges: {len(merges)}\n")
    f.write('\n'.join(log_lines))
print("Saved genre_merge_log.txt with only merges.")

# Final statistics
final_canonicals = set(raw_to_canonical.values())
print(f"Final unique canonical genres: {len(final_canonicals)}")
print(f"Reduction: {len(raw_genres)} -> {len(final_canonicals)}")

Loading artist_genres.csv...
Unique raw genres: 1997
Clustering normalized genres...
Number of fuzzy clusters: 1927
Total merges (raw changed): 70
Saved genre_clean_mapping.csv
Saved genre_merge_log.txt with only merges.
Final unique canonical genres: 1927
Reduction: 1997 -> 1927


In [25]:
import pandas as pd

# Load data
genres = pd.read_csv('artist_genres.csv')         # columns: artist_id, genre
mapping = pd.read_csv('genre_clean_mapping.csv')  # columns: raw_genre, canonical_genre

# Get unique raw genres from artist_genres
raw_in_data = set(genres['genre'].dropna().unique())

# Get raw genres that are covered by the mapping
mapped_raws = set(mapping['raw_genre'])

# Find missing
missing = raw_in_data - mapped_raws

if missing:
    print(f"Missing mappings: {len(missing)}")
    for g in sorted(missing)[:50]:   # show first 50
        print(f"  '{g}'")
else:
    print("All raw genres in artist_genres.csv are present in genre_clean_mapping.csv.")

All raw genres in artist_genres.csv are present in genre_clean_mapping.csv.


In [26]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ---------------------------------------------------------------------------
# 1. Load the necessary files
# ---------------------------------------------------------------------------
artist_genres = pd.read_csv('artist_genres.csv')              # artist_id, genre
genre_mapping = pd.read_csv('genre_clean_mapping.csv')        # raw_genre, canonical_genre
artist_dedup  = pd.read_csv('artist_dedup_mapping.csv')      # original_id, canonical_id

print(f"Loaded {len(artist_genres)} artist‑genre pairs.")
print(f"Genre mapping contains {len(genre_mapping)} entries.")
print(f"Artist dedup mapping contains {len(artist_dedup)} entries.")

# ---------------------------------------------------------------------------
# 2. Map original artist IDs to canonical artist IDs
# ---------------------------------------------------------------------------
id_to_canonical = dict(zip(artist_dedup['original_id'], artist_dedup['canonical_id']))

# Replace artist_id in the genres DataFrame with canonical ID
artist_genres['canonical_artist_id'] = artist_genres['artist_id'].map(id_to_canonical)

# If any original artist ID is missing from the mapping (shouldn't happen, but just in case),
# keep the original ID.
missing_ids = artist_genres['canonical_artist_id'].isna()
if missing_ids.any():
    print(f"Warning: {missing_ids.sum()} artist IDs not in dedup mapping – keeping original.")
    artist_genres.loc[missing_ids, 'canonical_artist_id'] = artist_genres.loc[missing_ids, 'artist_id']

# ---------------------------------------------------------------------------
# 3. Map raw genres to canonical genres
# ---------------------------------------------------------------------------
raw_to_canonical = dict(zip(genre_mapping['raw_genre'], genre_mapping['canonical_genre']))
artist_genres['canonical_genre'] = artist_genres['genre'].map(raw_to_canonical)

# Check for unmapped genres (should be none after our check)
unmapped = artist_genres['canonical_genre'].isna()
if unmapped.any():
    print(f"Warning: {unmapped.sum()} genres could not be mapped – dropping those rows.")
    artist_genres = artist_genres[~unmapped]

# ---------------------------------------------------------------------------
# 4. Build binary matrix
# ---------------------------------------------------------------------------
# Encode canonical artist IDs and canonical genres
artist_enc = LabelEncoder()
genre_enc  = LabelEncoder()

artist_idx = artist_enc.fit_transform(artist_genres['canonical_artist_id'])
genre_idx  = genre_enc.fit_transform(artist_genres['canonical_genre'])

n_artists = len(artist_enc.classes_)
n_genres  = len(genre_enc.classes_)

print(f"Canonical artists in feature matrix: {n_artists}")
print(f"Canonical genres in feature matrix: {n_genres}")

# Create binary matrix (dense, should be manageable with ~5000 x ~1500)
feature_matrix = np.zeros((n_artists, n_genres), dtype=np.float32)
feature_matrix[artist_idx, genre_idx] = 1.0

# Density
density = feature_matrix.sum() / (n_artists * n_genres) * 100
print(f"Matrix density: {density:.2f}%")

# ---------------------------------------------------------------------------
# 5. Save outputs
# ---------------------------------------------------------------------------
np.save('artist_genre_matrix.npy', feature_matrix)
print("Saved artist_genre_matrix.npy")

# Save the list of canonical artist IDs and genre names for later reference
pd.Series(artist_enc.classes_, name='canonical_artist_id').to_csv('artist_ids_for_matrix.csv', index=False)
pd.Series(genre_enc.classes_, name='canonical_genre').to_csv('genre_names_for_matrix.csv', index=False)
print("Saved artist_ids_for_matrix.csv and genre_names_for_matrix.csv")

# Quick sanity: how many artists have at least one genre?
artists_with_genres = (feature_matrix.sum(axis=1) > 0).sum()
print(f"Artists with at least one genre: {artists_with_genres} ({artists_with_genres/n_artists*100:.1f}%)")

Loaded 15890 artist‑genre pairs.
Genre mapping contains 1997 entries.
Artist dedup mapping contains 5234 entries.
Canonical artists in feature matrix: 3426
Canonical genres in feature matrix: 1935
Matrix density: 0.24%
Saved artist_genre_matrix.npy
Saved artist_ids_for_matrix.csv and genre_names_for_matrix.csv
Artists with at least one genre: 3426 (100.0%)


In [27]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
MIN_ARTIST_COUNT = 5          # keep only genres with ≥ this many artists

# ---------------------------------------------------------------------------
# 1. Load existing data
# ---------------------------------------------------------------------------
# Dense matrix (3426 artists × 1935 genres)
dense_matrix = np.load('artist_genre_matrix.npy')

# Lists of artist IDs and genre names corresponding to the matrix rows/cols
artist_ids_in_matrix = pd.read_csv('artist_ids_for_matrix.csv')['canonical_artist_id'].tolist()
genre_names_in_matrix = pd.read_csv('genre_names_for_matrix.csv')['canonical_genre'].tolist()

# Full set of canonical artists (from dedup mapping)
artist_dedup = pd.read_csv('artist_dedup_mapping.csv')   # original_id, canonical_id
all_canonical_ids = sorted(artist_dedup['canonical_id'].unique())
n_all_artists = len(all_canonical_ids)   # should be 5152

print(f"Current matrix shape: {dense_matrix.shape}")
print(f"Full canonical artists: {n_all_artists}")

# ---------------------------------------------------------------------------
# 2. Pad matrix to full artist set
# ---------------------------------------------------------------------------
# Map existing artist IDs to row indices
artist_to_row = {aid: i for i, aid in enumerate(artist_ids_in_matrix)}

# Build new dense matrix with all artists (zero for missing artists)
new_dense = np.zeros((n_all_artists, dense_matrix.shape[1]), dtype=np.float32)
for i, aid in enumerate(all_canonical_ids):
    if aid in artist_to_row:
        new_dense[i, :] = dense_matrix[artist_to_row[aid], :]

print(f"Padded matrix shape: {new_dense.shape}")

# ---------------------------------------------------------------------------
# 3. Filter low-frequency genres
# ---------------------------------------------------------------------------
genre_counts = new_dense.sum(axis=0)                # number of artists per genre
genre_mask = genre_counts >= MIN_ARTIST_COUNT
filtered_matrix = new_dense[:, genre_mask]

# Update genre names list
filtered_genre_names = [genre_names_in_matrix[i] for i in range(len(genre_names_in_matrix)) if genre_mask[i]]

print(f"Genres kept: {filtered_matrix.shape[1]} (removed {len(genre_names_in_matrix) - filtered_matrix.shape[1]} rare genres)")
print(f"Matrix density: {filtered_matrix.sum() / (filtered_matrix.shape[0] * filtered_matrix.shape[1]) * 100:.2f}%")

# ---------------------------------------------------------------------------
# 4. Convert to CSR and save
# ---------------------------------------------------------------------------
sparse_matrix = csr_matrix(filtered_matrix)

# Save sparse matrix (using npz which stores the CSR arrays)
from scipy.sparse import save_npz
save_npz('artist_genre_matrix_final.npz', sparse_matrix)
print("Saved artist_genre_matrix_final.npz")

# Save the ordered list of canonical artist IDs (all 5152)
pd.Series(all_canonical_ids, name='canonical_artist_id').to_csv('canonical_artist_ids.csv', index=False)

# Save the filtered genre names
pd.Series(filtered_genre_names, name='canonical_genre').to_csv('canonical_genre_names.csv', index=False)

print("Saved canonical_artist_ids.csv and canonical_genre_names.csv")

Current matrix shape: (3426, 1935)
Full canonical artists: 5152
Padded matrix shape: (5152, 1935)
Genres kept: 361 (removed 1574 rare genres)
Matrix density: 0.73%
Saved artist_genre_matrix_final.npz
Saved canonical_artist_ids.csv and canonical_genre_names.csv


In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from scipy.sparse import csr_matrix, load_npz
from sklearn.preprocessing import LabelEncoder
from collections import defaultdict
import random
import os
import pickle
import json

SEED = 43
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------
POSITIVE_SCORE_THRESHOLD = 1.0
TEST_PERCENT = 0.2
EMBEDDING_DIM = 200
BATCH_SIZE = 64
EPOCHS = 100
LR = 0.001
WEIGHT_DECAY = 1e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EXPORT_DIR = 'model_params'
REG_LAMBDA = 0.05          # weight of related-artist loss

# -------------------------------------------------------------------
# 1. Load data tables
# -------------------------------------------------------------------
def load_data(track_reactions_path, tracks_path, artists_path,
              artist_reactions_path, reaction_types_path):
    df_tr = pd.read_csv(track_reactions_path)
    df_tracks = pd.read_csv(tracks_path)
    df_artists = pd.read_csv(artists_path)
    df_ar = pd.read_csv(artist_reactions_path)
    df_rt = pd.read_csv(reaction_types_path)
    return df_tr, df_tracks, df_artists, df_ar, df_rt

def build_score_map(df_rt):
    return dict(zip(df_rt['id'], df_rt['score']))

# -------------------------------------------------------------------
# 2. Load canonical ID mapping and apply to artist references
# -------------------------------------------------------------------
def load_canonical_mapping(path='artist_dedup_mapping.csv'):
    dedup = pd.read_csv(path)
    mapping = dict(zip(dedup['original_id'], dedup['canonical_id']))
    return {int(k): int(v) for k, v in mapping.items()}

def to_canonical_artist(artist_id_str, mapping):
    """Convert a possibly comma-separated string of artist IDs to canonical IDs.
       Returns first canonical ID (primary artist) or -1 if empty."""
    if pd.isna(artist_id_str) or str(artist_id_str).strip() == '':
        return -1
    first = int(str(artist_id_str).split(',')[0].strip())
    return mapping.get(first, -1)

# -------------------------------------------------------------------
# 3. Build user-artist interaction matrix using canonical IDs
# -------------------------------------------------------------------
def build_user_artist_matrix(df_tr, df_tracks, df_ar, score_map, mapping):
    track_to_artist = {}
    for _, row in df_tracks.iterrows():
        tid = row['id']
        canonical = to_canonical_artist(row['artists_id'], mapping)
        track_to_artist[tid] = canonical

    interactions = set()

    # Track reactions
    for _, row in df_tr.iterrows():
        uid = row['user_id']
        tid = row['track_id']
        reaction_id = row['reaction_id']
        score = score_map.get(reaction_id, 0.0)
        if score > POSITIVE_SCORE_THRESHOLD:
            artist = track_to_artist.get(tid, -1)
            if artist != -1:
                interactions.add((uid, artist))

    # Artist reactions
    for _, row in df_ar.iterrows():
        uid = row['user_id']
        artist_id = row['artist_id']
        canonical = mapping.get(artist_id, -1)
        if canonical != -1:
            reaction_id = row['reaction_id']
            score = score_map.get(reaction_id, 0.0)
            if score > POSITIVE_SCORE_THRESHOLD:
                interactions.add((uid, canonical))

    # Uploads
    for _, row in df_tracks.iterrows():
        uploader_str = row['uploaded_by']
        if pd.isna(uploader_str):
            continue
        tid = row['id']
        artist = track_to_artist.get(tid, -1)
        if artist == -1:
            continue
        for uid_str in str(uploader_str).split(','):
            uid_str = uid_str.strip()
            if uid_str:
                uid = int(uid_str)
                interactions.add((uid, artist))

    # Encode
    users, artists = zip(*interactions)
    user_enc = LabelEncoder()
    artist_enc = LabelEncoder()
    user_idx = user_enc.fit_transform(users)
    artist_idx = artist_enc.fit_transform(artists)

    n_users = len(user_enc.classes_)
    n_artists = len(artist_enc.classes_)

    mat = csr_matrix((np.ones(len(interactions), dtype=np.float32),
                      (user_idx, artist_idx)),
                     shape=(n_users, n_artists))
    user_id_to_idx = {uid: i for i, uid in enumerate(user_enc.classes_)}
    return mat, user_id_to_idx, artist_enc, track_to_artist, user_enc

# -------------------------------------------------------------------
# 4. Build track-level split
# -------------------------------------------------------------------
def build_track_split(df_tr, df_tracks, score_map, test_percent=0.2):
    interactions = []
    for _, row in df_tr.iterrows():
        uid = row['user_id']
        tid = row['track_id']
        reaction_id = row['reaction_id']
        score = score_map.get(reaction_id, 0.0)
        if score > POSITIVE_SCORE_THRESHOLD:
            interactions.append((uid, tid))

    user_enc = LabelEncoder()
    item_enc = LabelEncoder()
    u_idx = user_enc.fit_transform([x[0] for x in interactions])
    i_idx = item_enc.fit_transform([x[1] for x in interactions])

    mat = csr_matrix((np.ones(len(interactions)), (u_idx, i_idx)),
                     shape=(len(user_enc.classes_), len(item_enc.classes_)))

    coo = mat.tocoo()
    np.random.seed(SEED)
    mask = np.random.rand(len(coo.data)) < (1 - test_percent)
    train = csr_matrix((coo.data[mask], (coo.row[mask], coo.col[mask])), shape=mat.shape)
    test  = csr_matrix((coo.data[~mask], (coo.row[~mask], coo.col[~mask])), shape=mat.shape)

    test_user_tracks = defaultdict(set)
    test_coo = test.tocoo()
    for u, i, v in zip(test_coo.row, test_coo.col, test_coo.data):
        if v > 0:
            uid = user_enc.classes_[u]
            tid = item_enc.classes_[i]
            test_user_tracks[uid].add(tid)

    track_pop = np.array(mat.sum(axis=0)).flatten()
    track_id_to_idx = {tid: i for i, tid in enumerate(item_enc.classes_)}

    seen_tracks = defaultdict(set)
    train_coo = train.tocoo()
    for u, i in zip(train_coo.row, train_coo.col):
        uid = user_enc.classes_[u]
        tid = item_enc.classes_[i]
        seen_tracks[uid].add(tid)

    return test_user_tracks, seen_tracks, track_pop, track_id_to_idx

# -------------------------------------------------------------------
# 5. BPR Dataset
# -------------------------------------------------------------------
class BPRDataset(Dataset):
    def __init__(self, mat, num_neg=1):
        coo = mat.tocoo()
        self.users = torch.LongTensor(coo.row)
        self.pos_items = torch.LongTensor(coo.col)
        self.n_items = mat.shape[1]
        self.num_neg = num_neg

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        user = self.users[idx]
        pos = self.pos_items[idx]
        negs = []
        while len(negs) < self.num_neg:
            neg = random.randint(0, self.n_items - 1)
            if neg != pos:
                negs.append(neg)
        return user, pos, torch.LongTensor(negs)

# -------------------------------------------------------------------
# 6. Hybrid model with genre features
# -------------------------------------------------------------------
class HybridArtistMF(nn.Module):
    def __init__(self, n_users, n_artists, emb_dim, genre_dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.artist_id_emb = nn.Embedding(n_artists, emb_dim)
        self.genre_proj = nn.Linear(genre_dim, emb_dim, bias=False)
        self.user_bias = nn.Embedding(n_users, 1)
        self.artist_bias = nn.Embedding(n_artists, 1)

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.artist_id_emb.weight)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.artist_bias.weight)

    def forward(self, user_idx, artist_idx, genre_features):
        u = self.user_emb(user_idx)
        a_id = self.artist_id_emb(artist_idx)
        a_genre = self.genre_proj(genre_features[artist_idx])
        a = a_id + a_genre
        dot = (u * a).sum(dim=-1)
        u_bias = self.user_bias(user_idx).squeeze(-1)
        a_bias = self.artist_bias(artist_idx).squeeze(-1)
        return dot + u_bias + a_bias

    def get_artist_embeddings(self, genre_features):
        with torch.no_grad():
            a_id = self.artist_id_emb.weight
            a_genre = self.genre_proj(genre_features)
            return a_id + a_genre

# -------------------------------------------------------------------
# 7. BPR loss + related-artist regularization
# -------------------------------------------------------------------
def bpr_loss_with_edges(model, user, pos, neg, genre_feat):
    pos_score = model(user, pos, genre_feat)
    # neg is (batch_size, num_neg)
    batch_size, num_neg = neg.shape
    user_exp = user.unsqueeze(1).expand(-1, num_neg).reshape(-1)
    neg_flat = neg.reshape(-1)
    neg_score = model(user_exp, neg_flat, genre_feat).view(batch_size, num_neg)
    diff = pos_score.unsqueeze(1) - neg_score
    return -torch.log(torch.sigmoid(diff) + 1e-10).mean()

def edge_loss(model, edge_a, edge_b, genre_feat):
    a_emb = model.artist_id_emb(edge_a) + model.genre_proj(genre_feat[edge_a])
    b_emb = model.artist_id_emb(edge_b) + model.genre_proj(genre_feat[edge_b])
    return ((a_emb - b_emb) ** 2).mean()

# -------------------------------------------------------------------
# 8. Training
# -------------------------------------------------------------------
def train_hybrid_model(mat, genre_feat, artist_enc, edge_indices, epochs=200):
    n_users, n_artists = mat.shape
    genre_dim = genre_feat.shape[1]
    dataset = BPRDataset(mat, num_neg=1)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    model = HybridArtistMF(n_users, n_artists, EMBEDDING_DIM, genre_dim).to(DEVICE)
    genre_feat = genre_feat.to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    edge_tensor = torch.LongTensor(edge_indices).to(DEVICE)  # (2, num_edges)

    for epoch in range(1, epochs+1):
        model.train()
        total_bpr = 0
        total_edge = 0
        batches = 0
        for user, pos, negs in dataloader:
            user = user.to(DEVICE)
            pos = pos.to(DEVICE)
            negs = negs.to(DEVICE)                     # keep 2D shape (batch_size, 1)

            optimizer.zero_grad()
            loss_bpr = bpr_loss_with_edges(model, user, pos, negs, genre_feat)

            # Sample edges for this batch
            if edge_tensor.shape[1] > 0:
                num_edges = min(len(user), edge_tensor.shape[1])
                idx = torch.randperm(edge_tensor.shape[1])[:num_edges]
                e_a = edge_tensor[0, idx]
                e_b = edge_tensor[1, idx]
                loss_edge = edge_loss(model, e_a, e_b, genre_feat)
            else:
                loss_edge = 0.0

            loss = loss_bpr + REG_LAMBDA * loss_edge
            loss.backward()
            optimizer.step()

            total_bpr += loss_bpr.item()
            total_edge += loss_edge.item() if isinstance(loss_edge, float) else loss_edge.item()
            batches += 1

        scheduler.step()
        if epoch % 5 == 0 or epoch == epochs:
            print(f"Epoch {epoch:03d}  BPR loss={total_bpr/batches:.4f}  "
                  f"Edge loss={total_edge/batches:.6f}")

    model.eval()
    with torch.no_grad():
        user_emb = model.user_emb.weight.cpu().numpy()
        artist_emb = model.get_artist_embeddings(genre_feat).cpu().numpy()
    return model, user_emb, artist_emb

# -------------------------------------------------------------------
# 9. Evaluation
# -------------------------------------------------------------------
def evaluate_track_recall_and_precision(user_emb, artist_emb, user_id_to_idx, artist_enc,
                                        track_to_artist, test_user_tracks, seen_tracks,
                                        track_pop, track_id_to_idx, top_k=10,
                                        n_artists=5, tracks_per_artist=5):
    artist_to_tracks = defaultdict(list)
    for tid, artist_id in track_to_artist.items():
        if artist_id in artist_enc.classes_ and tid in track_id_to_idx:
            aidx = artist_enc.transform([artist_id])[0]
            artist_to_tracks[aidx].append(tid)

    recalls = []
    precisions = []
    for uid, test_tids in test_user_tracks.items():
        if uid not in user_id_to_idx:
            continue
        u = user_id_to_idx[uid]
        scores = user_emb[u].dot(artist_emb.T)
        top_artists = np.argpartition(scores, -n_artists)[-n_artists:]
        top_artists = top_artists[np.argsort(scores[top_artists])[::-1]]

        seen = seen_tracks.get(uid, set())
        candidates = []
        for aidx in top_artists:
            track_list = artist_to_tracks.get(aidx, [])
            track_list_sorted = sorted(track_list, key=lambda t: track_pop[track_id_to_idx[t]], reverse=True)
            for t in track_list_sorted[:tracks_per_artist]:
                if t not in seen:
                    candidates.append((t, track_pop[track_id_to_idx[t]]))
        uniq = {}
        for t, pop in candidates:
            uniq[t] = pop
        candidates = sorted(uniq.items(), key=lambda x: x[1], reverse=True)
        top_tracks = [t for t,_ in candidates[:top_k]]

        hit = len(set(top_tracks) & test_tids)
        recall = hit / len(test_tids) if test_tids else 0.0
        precision = hit / top_k
        recalls.append(recall)
        precisions.append(precision)

    return np.mean(recalls) if recalls else 0.0, np.mean(precisions) if precisions else 0.0

# -------------------------------------------------------------------
# 10. Main
# -------------------------------------------------------------------
def main():
    # Paths
    df_tr, df_tracks, df_artists, df_ar, df_rt = load_data(
        'processed/track_reactions.csv',
        'processed/tracks.csv',
        'processed/artists_1.csv',
        'processed/artist_reactions.csv',
        'processed/reaction_types.csv'
    )
    score_map = build_score_map(df_rt)

    # Canonical ID mapping
    print("Loading canonical artist mapping...")
    mapping = load_canonical_mapping('artist_dedup_mapping.csv')
    print(f"Canonical mapping covers {len(mapping)} original IDs.")

    # Build user-artist matrix with canonical IDs
    print("Building user-artist matrix (canonical IDs)...")
    ua_mat, user_id_to_idx, artist_enc, track_to_artist, user_enc = build_user_artist_matrix(
        df_tr, df_tracks, df_ar, score_map, mapping
    )
    print(f"User-Artist matrix: {ua_mat.shape[0]} users x {ua_mat.shape[1]} artists, {ua_mat.nnz} interactions")

    # Track split
    print("Building track test set...")
    test_user_tracks, seen_tracks, track_pop, track_id_to_idx = build_track_split(
        df_tr, df_tracks, score_map, TEST_PERCENT
    )

    # Load genre features and align to interaction artists
    print("Loading genre features...")
    genre_sparse = load_npz('artist_genre_matrix_final.npz')
    genre_feat_full = torch.FloatTensor(genre_sparse.toarray())
    print(f"Full genre feature matrix: {genre_feat_full.shape}")

    # Align to the artists in the interaction matrix
    canonical_ids = pd.read_csv('canonical_artist_ids.csv')['canonical_artist_id'].tolist()
    canonical_to_row = {cid: i for i, cid in enumerate(canonical_ids)}
    selected_rows = [canonical_to_row[artist_id] for artist_id in artist_enc.classes_]
    genre_feat = genre_feat_full[selected_rows]   # shape: (n_artists_in_interaction, genre_dim)
    print(f"Aligned genre features for interaction artists: {genre_feat.shape}")

    # Load artist links and prepare edge indices
    print("Loading related-artist edges...")
    links_df = pd.read_csv('artist_links.csv')
    edge_indices = []
    for _, row in links_df.iterrows():
        aid = row['artist_id']
        rid = row['related_artist_id']
        can_a = mapping.get(aid, aid)
        can_r = mapping.get(rid, rid)
        if can_a in artist_enc.classes_ and can_r in artist_enc.classes_:
            a_idx = artist_enc.transform([can_a])[0]
            r_idx = artist_enc.transform([can_r])[0]
            if a_idx != r_idx:
                edge_indices.append([a_idx, r_idx])
    edge_indices = np.array(edge_indices).T
    print(f"Valid related-artist edges: {edge_indices.shape[1]}")

    # Train hybrid model
    print("Training hybrid artist MF with genre features + related-artist loss...")
    model, user_emb, artist_emb = train_hybrid_model(ua_mat, genre_feat, artist_enc,
                                                     edge_indices, epochs=EPOCHS)

    # Evaluate
    print("Evaluating track recall & precision...")
    rec, prec = evaluate_track_recall_and_precision(
        user_emb, artist_emb, user_id_to_idx, artist_enc,
        track_to_artist, test_user_tracks, seen_tracks,
        track_pop, track_id_to_idx
    )
    print(f"Track Recall@10: {rec:.4f}   Precision@10: {prec:.4f}")

    # Export
    os.makedirs(EXPORT_DIR, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(EXPORT_DIR, 'model_state.pt'))
    with open(os.path.join(EXPORT_DIR, 'user_enc.pkl'), 'wb') as f:
        pickle.dump(user_enc, f)
    with open(os.path.join(EXPORT_DIR, 'artist_enc.pkl'), 'wb') as f:
        pickle.dump(artist_enc, f)
    with open(os.path.join(EXPORT_DIR, 'track_to_artist.pkl'), 'wb') as f:
        pickle.dump(track_to_artist, f)
    with open(os.path.join(EXPORT_DIR, 'track_id_to_idx.pkl'), 'wb') as f:
        pickle.dump(track_id_to_idx, f)
    np.save(os.path.join(EXPORT_DIR, 'track_pop.npy'), track_pop)
    np.save(os.path.join(EXPORT_DIR, 'user_embeddings.npy'), user_emb)
    np.save(os.path.join(EXPORT_DIR, 'artist_embeddings.npy'), artist_emb)
    torch.save(genre_feat, os.path.join(EXPORT_DIR, 'genre_features.pt'))
    config = {
        'embedding_dim': EMBEDDING_DIM,
        'n_users': len(user_enc.classes_),
        'n_artists': len(artist_enc.classes_),
        'genre_dim': genre_feat.shape[1],
        'positive_score_threshold': POSITIVE_SCORE_THRESHOLD,
        'n_artists_rec': 5,
        'tracks_per_artist': 5,
        'top_k': 10
    }
    with open(os.path.join(EXPORT_DIR, 'config.json'), 'w') as f:
        json.dump(config, f, indent=2)

    print(f"Model and artifacts saved to {EXPORT_DIR}")

if __name__ == "__main__":
    main()

Loading canonical artist mapping...
Canonical mapping covers 5234 original IDs.
Building user-artist matrix (canonical IDs)...
User-Artist matrix: 1285 users x 4919 artists, 59959 interactions
Building track test set...
Loading genre features...
Full genre feature matrix: torch.Size([5152, 361])
Aligned genre features for interaction artists: torch.Size([4919, 361])
Loading related-artist edges...
Valid related-artist edges: 6654
Training hybrid artist MF with genre features + related-artist loss...
Epoch 005  BPR loss=0.2407  Edge loss=0.033241
Epoch 010  BPR loss=0.1675  Edge loss=0.043867
Epoch 015  BPR loss=0.1374  Edge loss=0.049422
Epoch 020  BPR loss=0.1227  Edge loss=0.051855
Epoch 025  BPR loss=0.1166  Edge loss=0.053205
Epoch 030  BPR loss=0.1115  Edge loss=0.054287
Epoch 035  BPR loss=0.1101  Edge loss=0.054130
Epoch 040  BPR loss=0.1040  Edge loss=0.054591
Epoch 045  BPR loss=0.1022  Edge loss=0.054616
Epoch 050  BPR loss=0.0981  Edge loss=0.054150
Epoch 055  BPR loss=0.096

In [4]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from scipy.sparse import csr_matrix, load_npz
from sklearn.preprocessing import LabelEncoder
from collections import defaultdict
import random
import os
import pickle
import json

# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------
SEED = 43
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

POSITIVE_SCORE_THRESHOLD = 1.0
TEST_PERCENT = 0.2
EMBEDDING_DIM = 200
BATCH_SIZE = 64
EPOCHS = 100
LR = 0.001
WEIGHT_DECAY = 1e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EXPORT_DIR = 'model_params'

UPLOAD_WEIGHT = 5.0                # weight for upload interactions
EDGE_WEIGHT = 0.2                  # weight for synthetic related-artist interactions
REG_LAMBDA = 0.05                  # related-artist edge loss coefficient
USE_SYNTHETIC_EDGES = True
ALPHA_PERSONAL = 0.3               # weight for personalised track score

# -------------------------------------------------------------------
# 1. Data loading
# -------------------------------------------------------------------
def load_data(track_reactions_path, tracks_path, artists_path,
              artist_reactions_path, reaction_types_path):
    df_tr = pd.read_csv(track_reactions_path)
    df_tracks = pd.read_csv(tracks_path)
    df_artists = pd.read_csv(artists_path)
    df_ar = pd.read_csv(artist_reactions_path)
    df_rt = pd.read_csv(reaction_types_path)
    return df_tr, df_tracks, df_artists, df_ar, df_rt

def build_score_map(df_rt):
    return dict(zip(df_rt['id'], df_rt['score']))

# -------------------------------------------------------------------
# 2. Canonical artist ID handling
# -------------------------------------------------------------------
def load_canonical_mapping(path='artist_dedup_mapping.csv'):
    dedup = pd.read_csv(path)
    mapping = {int(k): int(v) for k, v in zip(dedup['original_id'], dedup['canonical_id'])}
    return mapping

def to_canonical_artist(artist_id_str, mapping):
    if pd.isna(artist_id_str) or str(artist_id_str).strip() == '':
        return -1
    first = int(str(artist_id_str).split(',')[0].strip())
    return mapping.get(first, -1)

# -------------------------------------------------------------------
# 3. Build weighted user‑artist matrix
# -------------------------------------------------------------------
def build_user_artist_matrix(df_tr, df_tracks, df_ar, score_map, mapping,
                             upload_weight=UPLOAD_WEIGHT):
    track_to_artist = {}
    for _, row in df_tracks.iterrows():
        tid = row['id']
        canonical = to_canonical_artist(row['artists_id'], mapping)
        track_to_artist[tid] = canonical

    pair_weight = defaultdict(float)

    # Track reactions
    for _, row in df_tr.iterrows():
        uid = row['user_id']
        tid = row['track_id']
        reaction_id = row['reaction_id']
        score = score_map.get(reaction_id, 0.0)
        if score > POSITIVE_SCORE_THRESHOLD:
            artist = track_to_artist.get(tid, -1)
            if artist != -1:
                pair_weight[(uid, artist)] += score

    # Artist reactions
    for _, row in df_ar.iterrows():
        uid = row['user_id']
        artist_id = row['artist_id']
        canonical = mapping.get(artist_id, -1)
        if canonical != -1:
            reaction_id = row['reaction_id']
            score = score_map.get(reaction_id, 0.0)
            if score > POSITIVE_SCORE_THRESHOLD:
                pair_weight[(uid, canonical)] += score

    # Uploads (strong weight)
    for _, row in df_tracks.iterrows():
        uploader_str = row['uploaded_by']
        if pd.isna(uploader_str):
            continue
        tid = row['id']
        artist = track_to_artist.get(tid, -1)
        if artist == -1:
            continue
        for uid_str in str(uploader_str).split(','):
            uid_str = uid_str.strip()
            if uid_str:
                uid = int(uid_str)
                pair_weight[(uid, artist)] += upload_weight

    # Keep pairs with positive weight
    pairs = [(u, a) for (u, a), w in pair_weight.items() if w > 0]
    users, artists = zip(*pairs)
    weights = [pair_weight[(u, a)] for (u, a) in pairs]

    user_enc = LabelEncoder()
    artist_enc = LabelEncoder()
    user_idx = user_enc.fit_transform(users)
    artist_idx = artist_enc.fit_transform(artists)

    n_users = len(user_enc.classes_)
    n_artists = len(artist_enc.classes_)

    mat = csr_matrix((np.ones(len(pairs), dtype=np.float32), (user_idx, artist_idx)),
                     shape=(n_users, n_artists))
    weight_mat = csr_matrix((weights, (user_idx, artist_idx)), shape=(n_users, n_artists))

    user_id_to_idx = {uid: i for i, uid in enumerate(user_enc.classes_)}
    return mat, weight_mat, user_id_to_idx, artist_enc, track_to_artist, user_enc

# -------------------------------------------------------------------
# 4. Add synthetic related‑artist interactions
# -------------------------------------------------------------------
def add_related_artist_edges(mat, weight_mat, user_id_to_idx, artist_enc,
                             links_df, mapping, edge_weight=EDGE_WEIGHT):
    artist_to_related = defaultdict(list)
    for _, row in links_df.iterrows():
        aid = row['artist_id']
        rid = row['related_artist_id']
        can_a = mapping.get(aid, aid)
        can_r = mapping.get(rid, rid)
        if can_a in artist_enc.classes_ and can_r in artist_enc.classes_:
            a_idx = artist_enc.transform([can_a])[0]
            r_idx = artist_enc.transform([can_r])[0]
            if a_idx != r_idx:
                artist_to_related[a_idx].append(r_idx)

    mat_coo = mat.tocoo()
    new_rows, new_cols, new_weights = [], [], []
    for u, a in zip(mat_coo.row, mat_coo.col):
        if a in artist_to_related:
            for r in artist_to_related[a]:
                if mat[u, r] == 0:   # user hasn't interacted with related artist
                    new_rows.append(u)
                    new_cols.append(r)
                    new_weights.append(edge_weight)

    if new_rows:
        n_users, n_artists = mat.shape
        orig_rows = mat_coo.row
        orig_cols = mat_coo.col
        orig_data = mat_coo.data
        orig_weights = weight_mat.data

        all_rows = np.concatenate([orig_rows, new_rows])
        all_cols = np.concatenate([orig_cols, new_cols])
        all_data = np.concatenate([orig_data, np.ones(len(new_rows), dtype=np.float32)])
        all_weights = np.concatenate([orig_weights, new_weights])

        mat_new = csr_matrix((all_data, (all_rows, all_cols)), shape=(n_users, n_artists))
        weight_mat_new = csr_matrix((all_weights, (all_rows, all_cols)), shape=(n_users, n_artists))
        return mat_new, weight_mat_new
    return mat, weight_mat

# -------------------------------------------------------------------
# 5. Build track‑level split and reactor sets
# -------------------------------------------------------------------
def build_track_split(df_tr, df_tracks, score_map, test_percent=0.2):
    interactions = []
    for _, row in df_tr.iterrows():
        uid = row['user_id']
        tid = row['track_id']
        reaction_id = row['reaction_id']
        score = score_map.get(reaction_id, 0.0)
        if score > POSITIVE_SCORE_THRESHOLD:
            interactions.append((uid, tid))

    user_enc = LabelEncoder()
    item_enc = LabelEncoder()
    u_idx = user_enc.fit_transform([x[0] for x in interactions])
    i_idx = item_enc.fit_transform([x[1] for x in interactions])

    mat = csr_matrix((np.ones(len(interactions)), (u_idx, i_idx)),
                     shape=(len(user_enc.classes_), len(item_enc.classes_)))

    coo = mat.tocoo()
    np.random.seed(SEED)
    mask = np.random.rand(len(coo.data)) < (1 - test_percent)
    train = csr_matrix((coo.data[mask], (coo.row[mask], coo.col[mask])), shape=mat.shape)
    test  = csr_matrix((coo.data[~mask], (coo.row[~mask], coo.col[~mask])), shape=mat.shape)

    test_user_tracks = defaultdict(set)
    test_coo = test.tocoo()
    for u, i, v in zip(test_coo.row, test_coo.col, test_coo.data):
        if v > 0:
            uid = user_enc.classes_[u]
            tid = item_enc.classes_[i]
            test_user_tracks[uid].add(tid)

    track_pop = np.array(mat.sum(axis=0)).flatten()
    track_id_to_idx = {tid: i for i, tid in enumerate(item_enc.classes_)}

    seen_tracks = defaultdict(set)
    train_coo = train.tocoo()
    for u, i in zip(train_coo.row, train_coo.col):
        uid = user_enc.classes_[u]
        tid = item_enc.classes_[i]
        seen_tracks[uid].add(tid)

    # Build reactor sets for each track (user indices)
    track_reactors = defaultdict(set)
    for u, i in zip(train_coo.row, train_coo.col):
        track_reactors[i].add(u)

    return test_user_tracks, seen_tracks, track_pop, track_id_to_idx, track_reactors

# -------------------------------------------------------------------
# 6. Datasets and model
# -------------------------------------------------------------------
class BPRDataset(Dataset):
    def __init__(self, mat, weight_mat, num_neg=1):
        coo = mat.tocoo()
        self.users = torch.LongTensor(coo.row)
        self.pos_items = torch.LongTensor(coo.col)
        # match weights to the order of coo entries
        self.weights = torch.FloatTensor(weight_mat[coo.row, coo.col].A1)
        self.n_items = mat.shape[1]
        self.num_neg = num_neg

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        user = self.users[idx]
        pos = self.pos_items[idx]
        w = self.weights[idx]
        negs = []
        while len(negs) < self.num_neg:
            neg = random.randint(0, self.n_items - 1)
            if neg != pos:
                negs.append(neg)
        return user, pos, w, torch.LongTensor(negs)

class HybridArtistMF(nn.Module):
    def __init__(self, n_users, n_artists, emb_dim, genre_dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.artist_id_emb = nn.Embedding(n_artists, emb_dim)
        self.genre_proj = nn.Linear(genre_dim, emb_dim, bias=False)
        self.user_bias = nn.Embedding(n_users, 1)
        self.artist_bias = nn.Embedding(n_artists, 1)

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.artist_id_emb.weight)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.artist_bias.weight)

    def forward(self, user_idx, artist_idx, genre_features):
        u = self.user_emb(user_idx)
        a_id = self.artist_id_emb(artist_idx)
        a_genre = self.genre_proj(genre_features[artist_idx])
        a = a_id + a_genre
        dot = (u * a).sum(dim=-1)
        u_bias = self.user_bias(user_idx).squeeze(-1)
        a_bias = self.artist_bias(artist_idx).squeeze(-1)
        return dot + u_bias + a_bias

    def get_artist_embeddings(self, genre_features):
        with torch.no_grad():
            a_id = self.artist_id_emb.weight
            a_genre = self.genre_proj(genre_features)
            return a_id + a_genre

# -------------------------------------------------------------------
# 7. Loss functions
# -------------------------------------------------------------------
def weighted_bpr_loss(model, user, pos, neg, w, genre_feat):
    pos_score = model(user, pos, genre_feat)
    batch_size, num_neg = neg.shape
    user_exp = user.unsqueeze(1).expand(-1, num_neg).reshape(-1)
    neg_flat = neg.reshape(-1)
    neg_score = model(user_exp, neg_flat, genre_feat).view(batch_size, num_neg)
    diff = pos_score.unsqueeze(1) - neg_score
    bpr = -torch.log(torch.sigmoid(diff) + 1e-10).mean(dim=1)   # per‑example loss
    return (w * bpr).mean()

def edge_loss(model, edge_a, edge_b, genre_feat):
    a_emb = model.artist_id_emb(edge_a) + model.genre_proj(genre_feat[edge_a])
    b_emb = model.artist_id_emb(edge_b) + model.genre_proj(genre_feat[edge_b])
    return ((a_emb - b_emb) ** 2).mean()

# -------------------------------------------------------------------
# 8. Training
# -------------------------------------------------------------------
def train_hybrid_model(mat, weight_mat, genre_feat, artist_enc, edge_indices, epochs=200):
    n_users, n_artists = mat.shape
    genre_dim = genre_feat.shape[1]
    dataset = BPRDataset(mat, weight_mat, num_neg=1)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    model = HybridArtistMF(n_users, n_artists, EMBEDDING_DIM, genre_dim).to(DEVICE)
    genre_feat = genre_feat.to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    edge_tensor = torch.LongTensor(edge_indices).to(DEVICE)

    for epoch in range(1, epochs+1):
        model.train()
        total_bpr = 0.0
        total_edge = 0.0
        batches = 0
        for user, pos, w, negs in dataloader:
            user, pos, w, negs = user.to(DEVICE), pos.to(DEVICE), w.to(DEVICE), negs.to(DEVICE)

            optimizer.zero_grad()
            bpr = weighted_bpr_loss(model, user, pos, negs, w, genre_feat)

            if edge_tensor.shape[1] > 0:
                num_edges = min(len(user), edge_tensor.shape[1])
                idx = torch.randperm(edge_tensor.shape[1])[:num_edges]
                e_a, e_b = edge_tensor[0, idx], edge_tensor[1, idx]
                e_loss = edge_loss(model, e_a, e_b, genre_feat)
            else:
                e_loss = 0.0

            loss = bpr + REG_LAMBDA * e_loss
            loss.backward()
            optimizer.step()

            total_bpr += bpr.item()
            total_edge += e_loss.item() if isinstance(e_loss, torch.Tensor) else e_loss
            batches += 1

        scheduler.step()
        if epoch % 5 == 0 or epoch == epochs:
            print(f"Epoch {epoch:03d}  BPR loss={total_bpr/batches:.4f}  "
                  f"Edge loss={total_edge/batches:.6f}")

    model.eval()
    with torch.no_grad():
        user_emb = model.user_emb.weight.cpu().numpy()
        artist_emb = model.get_artist_embeddings(genre_feat).cpu().numpy()
    return model, user_emb, artist_emb

# -------------------------------------------------------------------
# 9. Evaluation with personalised track scoring
# -------------------------------------------------------------------
def evaluate_recall_precision(user_emb, artist_emb, user_id_to_idx, artist_enc,
                              track_to_artist, test_user_tracks, seen_tracks,
                              track_pop, track_id_to_idx, track_reactors,
                              ua_mat, top_k=10, n_artists=5, tracks_per_artist=5,
                              alpha=ALPHA_PERSONAL):
    # Map artist index -> list of track ids
    artist_to_tracks = defaultdict(list)
    for tid, artist_id in track_to_artist.items():
        if artist_id in artist_enc.classes_ and tid in track_id_to_idx:
            aidx = artist_enc.transform([artist_id])[0]
            artist_to_tracks[aidx].append(tid)

    # Taste neighbours: for each user, set of other users who share at least one liked artist
    ua_coo = ua_mat.tocoo()
    artist_to_users = defaultdict(set)
    user_to_artists = defaultdict(set)
    for u, a in zip(ua_coo.row, ua_coo.col):
        artist_to_users[a].add(u)
        user_to_artists[u].add(a)

    user_neighbors = {}
    for u in user_to_artists:
        neighbors = set()
        for a in user_to_artists[u]:
            neighbors.update(artist_to_users[a])
        neighbors.discard(u)
        user_neighbors[u] = neighbors

    recalls, precisions = [], []
    for uid, test_tids in test_user_tracks.items():
        if uid not in user_id_to_idx:
            continue
        u = user_id_to_idx[uid]
        scores = user_emb[u].dot(artist_emb.T)
        top_artists = np.argpartition(scores, -n_artists)[-n_artists:]
        top_artists = top_artists[np.argsort(scores[top_artists])[::-1]]

        seen = seen_tracks.get(uid, set())
        neighbors = user_neighbors.get(u, set())

        candidates = []
        for aidx in top_artists:
            for t in artist_to_tracks.get(aidx, []):
                if t in seen:
                    continue
                tidx = track_id_to_idx[t]
                pop = track_pop[tidx]
                # Personalised score: overlap between track's reactors and user's neighbours
                reactors = track_reactors.get(tidx, set())
                if neighbors and reactors:
                    overlap = len(reactors & neighbors)
                    pers = overlap / len(reactors)   # fraction of reactors that are neighbours
                else:
                    pers = 0.0
                final_score = pop + alpha * pers * max(pop, 1)
                candidates.append((t, final_score))

        # Deduplicate and sort
        uniq = {}
        for t, score in candidates:
            if t not in uniq or score > uniq[t]:
                uniq[t] = score
        top_tracks = [t for t, _ in sorted(uniq.items(), key=lambda x: x[1], reverse=True)[:top_k]]

        hit = len(set(top_tracks) & test_tids)
        recall = hit / len(test_tids) if test_tids else 0.0
        precision = hit / top_k
        recalls.append(recall)
        precisions.append(precision)

    return np.mean(recalls) if recalls else 0.0, np.mean(precisions) if precisions else 0.0

# -------------------------------------------------------------------
# 10. Main
# -------------------------------------------------------------------
def main():
    # Load data
    df_tr, df_tracks, df_artists, df_ar, df_rt = load_data(
        'processed/track_reactions.csv',
        'processed/tracks.csv',
        'processed/artists_1.csv',
        'processed/artist_reactions.csv',
        'processed/reaction_types.csv'
    )
    score_map = build_score_map(df_rt)
    mapping = load_canonical_mapping('artist_dedup_mapping.csv')

    # User‑artist matrix
    print("Building weighted user‑artist matrix...")
    ua_mat, weight_mat, user_id_to_idx, artist_enc, track_to_artist, user_enc = build_user_artist_matrix(
        df_tr, df_tracks, df_ar, score_map, mapping
    )
    print(f"User‑Artist matrix: {ua_mat.shape[0]} users x {ua_mat.shape[1]} artists, {ua_mat.nnz} interactions")

    # Synthetic related‑artist edges
    if USE_SYNTHETIC_EDGES:
        links_df = pd.read_csv('artist_links.csv')
        print("Adding synthetic related‑artist interactions...")
        ua_mat, weight_mat = add_related_artist_edges(
            ua_mat, weight_mat, user_id_to_idx, artist_enc, links_df, mapping
        )
        print(f"After augmentation: {ua_mat.nnz} interactions")

    # Track split & reactor sets
    test_user_tracks, seen_tracks, track_pop, track_id_to_idx, track_reactors = build_track_split(
        df_tr, df_tracks, score_map, TEST_PERCENT
    )

    # Genre features (aligned)
    genre_sparse = load_npz('artist_genre_matrix_final.npz')
    genre_feat_full = torch.FloatTensor(genre_sparse.toarray())
    canonical_ids = pd.read_csv('canonical_artist_ids.csv')['canonical_artist_id'].tolist()
    canonical_to_row = {cid: i for i, cid in enumerate(canonical_ids)}
    selected_rows = [canonical_to_row[aid] for aid in artist_enc.classes_]
    genre_feat = genre_feat_full[selected_rows]

    # Related‑artist edges for regularisation loss
    links_df = pd.read_csv('artist_links.csv')
    edge_indices = []
    for _, row in links_df.iterrows():
        aid = row['artist_id']
        rid = row['related_artist_id']
        can_a = mapping.get(aid, aid)
        can_r = mapping.get(rid, rid)
        if can_a in artist_enc.classes_ and can_r in artist_enc.classes_:
            a_idx = artist_enc.transform([can_a])[0]
            r_idx = artist_enc.transform([can_r])[0]
            if a_idx != r_idx:
                edge_indices.append([a_idx, r_idx])
    edge_indices = np.array(edge_indices).T

    # Train
    print("Training hybrid model...")
    model, user_emb, artist_emb = train_hybrid_model(ua_mat, weight_mat, genre_feat, artist_enc,
                                                     edge_indices, epochs=EPOCHS)

    # Evaluate
    rec, prec = evaluate_recall_precision(
        user_emb, artist_emb, user_id_to_idx, artist_enc,
        track_to_artist, test_user_tracks, seen_tracks,
        track_pop, track_id_to_idx, track_reactors, ua_mat
    )
    print(f"Track Recall@10: {rec:.4f}   Precision@10: {prec:.4f}")

    # Export
    os.makedirs(EXPORT_DIR, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(EXPORT_DIR, 'model_state.pt'))
    with open(os.path.join(EXPORT_DIR, 'user_enc.pkl'), 'wb') as f:
        pickle.dump(user_enc, f)
    with open(os.path.join(EXPORT_DIR, 'artist_enc.pkl'), 'wb') as f:
        pickle.dump(artist_enc, f)
    with open(os.path.join(EXPORT_DIR, 'track_to_artist.pkl'), 'wb') as f:
        pickle.dump(track_to_artist, f)
    with open(os.path.join(EXPORT_DIR, 'track_id_to_idx.pkl'), 'wb') as f:
        pickle.dump(track_id_to_idx, f)
    np.save(os.path.join(EXPORT_DIR, 'track_pop.npy'), track_pop)
    np.save(os.path.join(EXPORT_DIR, 'user_embeddings.npy'), user_emb)
    np.save(os.path.join(EXPORT_DIR, 'artist_embeddings.npy'), artist_emb)
    torch.save(genre_feat, os.path.join(EXPORT_DIR, 'genre_features.pt'))

    config = {
        'embedding_dim': EMBEDDING_DIM,
        'n_users': len(user_enc.classes_),
        'n_artists': len(artist_enc.classes_),
        'genre_dim': genre_feat.shape[1],
        'positive_score_threshold': POSITIVE_SCORE_THRESHOLD,
        'n_artists_rec': 5,
        'tracks_per_artist': 5,
        'top_k': 10
    }
    with open(os.path.join(EXPORT_DIR, 'config.json'), 'w') as f:
        json.dump(config, f, indent=2)

    print(f"Model and artifacts saved to {EXPORT_DIR}")

if __name__ == "__main__":
    main()

Building weighted user‑artist matrix...
User‑Artist matrix: 1285 users x 4919 artists, 59959 interactions
Adding synthetic related‑artist interactions...
After augmentation: 145685 interactions
Training hybrid model...
Epoch 005  BPR loss=0.8007  Edge loss=0.047265
Epoch 010  BPR loss=0.5602  Edge loss=0.064830
Epoch 015  BPR loss=0.4931  Edge loss=0.073584
Epoch 020  BPR loss=0.4661  Edge loss=0.077632
Epoch 025  BPR loss=0.4369  Edge loss=0.079042
Epoch 030  BPR loss=0.4121  Edge loss=0.079831
Epoch 035  BPR loss=0.4027  Edge loss=0.079100
Epoch 040  BPR loss=0.3977  Edge loss=0.078784
Epoch 045  BPR loss=0.3678  Edge loss=0.077522
Epoch 050  BPR loss=0.3625  Edge loss=0.075651
Epoch 055  BPR loss=0.3491  Edge loss=0.074002
Epoch 060  BPR loss=0.3309  Edge loss=0.072297
Epoch 065  BPR loss=0.3257  Edge loss=0.070812
Epoch 070  BPR loss=0.3123  Edge loss=0.069564
Epoch 075  BPR loss=0.3104  Edge loss=0.068279
Epoch 080  BPR loss=0.3099  Edge loss=0.067103
Epoch 085  BPR loss=0.2933  E

In [6]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from scipy.sparse import csr_matrix, load_npz
from sklearn.preprocessing import LabelEncoder
from collections import defaultdict
import random
import os
import pickle
import json

# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------
SEED = 43
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

POSITIVE_SCORE_THRESHOLD = 1.0
TEST_PERCENT = 0.2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EXPORT_DIR = 'model_params'

# ----- Artist model hyperparameters -----
ARTIST_EMBEDDING_DIM = 200
ARTIST_BATCH_SIZE = 64
ARTIST_EPOCHS = 100
ARTIST_LR = 0.001
ARTIST_WEIGHT_DECAY = 1e-5
UPLOAD_WEIGHT = 5.0
EDGE_WEIGHT = 0.2
REG_LAMBDA = 0.05
USE_SYNTHETIC_EDGES = True

# ----- Track model hyperparameters -----
TRACK_EMBEDDING_DIM = 100
TRACK_BATCH_SIZE = 256
TRACK_EPOCHS = 100
TRACK_LR = 0.001
TRACK_WEIGHT_DECAY = 1e-5

# ----- Ensemble evaluation parameters -----
N_ARTISTS_CANDIDATES = 15          # number of top artists from artist model
TRACKS_PER_ARTIST_CANDIDATE = 10   # tracks per artist in candidate pool
TOP_K_FINAL = 10                   # final number of tracks to return
USE_ENSEMBLE = True                # set to False to use artist‑only track selection

# -------------------------------------------------------------------
# 1. Data loading
# -------------------------------------------------------------------
def load_data(track_reactions_path, tracks_path, artists_path,
              artist_reactions_path, reaction_types_path):
    df_tr = pd.read_csv(track_reactions_path)
    df_tracks = pd.read_csv(tracks_path)
    df_artists = pd.read_csv(artists_path)
    df_ar = pd.read_csv(artist_reactions_path)
    df_rt = pd.read_csv(reaction_types_path)
    return df_tr, df_tracks, df_artists, df_ar, df_rt

def build_score_map(df_rt):
    return dict(zip(df_rt['id'], df_rt['score']))

# -------------------------------------------------------------------
# 2. Canonical artist ID handling
# -------------------------------------------------------------------
def load_canonical_mapping(path='artist_dedup_mapping.csv'):
    dedup = pd.read_csv(path)
    mapping = {int(k): int(v) for k, v in zip(dedup['original_id'], dedup['canonical_id'])}
    return mapping

def to_canonical_artist(artist_id_str, mapping):
    if pd.isna(artist_id_str) or str(artist_id_str).strip() == '':
        return -1
    first = int(str(artist_id_str).split(',')[0].strip())
    return mapping.get(first, -1)

# -------------------------------------------------------------------
# 3. Build weighted user‑artist matrix
# -------------------------------------------------------------------
def build_user_artist_matrix(df_tr, df_tracks, df_ar, score_map, mapping,
                             upload_weight=UPLOAD_WEIGHT):
    track_to_artist = {}
    for _, row in df_tracks.iterrows():
        tid = row['id']
        canonical = to_canonical_artist(row['artists_id'], mapping)
        track_to_artist[tid] = canonical

    pair_weight = defaultdict(float)

    # Track reactions
    for _, row in df_tr.iterrows():
        uid = row['user_id']
        tid = row['track_id']
        reaction_id = row['reaction_id']
        score = score_map.get(reaction_id, 0.0)
        if score > POSITIVE_SCORE_THRESHOLD:
            artist = track_to_artist.get(tid, -1)
            if artist != -1:
                pair_weight[(uid, artist)] += score

    # Artist reactions
    for _, row in df_ar.iterrows():
        uid = row['user_id']
        artist_id = row['artist_id']
        canonical = mapping.get(artist_id, -1)
        if canonical != -1:
            reaction_id = row['reaction_id']
            score = score_map.get(reaction_id, 0.0)
            if score > POSITIVE_SCORE_THRESHOLD:
                pair_weight[(uid, canonical)] += score

    # Uploads
    for _, row in df_tracks.iterrows():
        uploader_str = row['uploaded_by']
        if pd.isna(uploader_str):
            continue
        tid = row['id']
        artist = track_to_artist.get(tid, -1)
        if artist == -1:
            continue
        for uid_str in str(uploader_str).split(','):
            uid_str = uid_str.strip()
            if uid_str:
                uid = int(uid_str)
                pair_weight[(uid, artist)] += upload_weight

    pairs = [(u, a) for (u, a), w in pair_weight.items() if w > 0]
    users, artists = zip(*pairs)
    weights = [pair_weight[(u, a)] for (u, a) in pairs]

    user_enc = LabelEncoder()
    artist_enc = LabelEncoder()
    user_idx = user_enc.fit_transform(users)
    artist_idx = artist_enc.fit_transform(artists)

    n_users = len(user_enc.classes_)
    n_artists = len(artist_enc.classes_)

    mat = csr_matrix((np.ones(len(pairs), dtype=np.float32), (user_idx, artist_idx)),
                     shape=(n_users, n_artists))
    weight_mat = csr_matrix((weights, (user_idx, artist_idx)), shape=(n_users, n_artists))

    user_id_to_idx = {uid: i for i, uid in enumerate(user_enc.classes_)}
    return mat, weight_mat, user_id_to_idx, artist_enc, track_to_artist, user_enc

# -------------------------------------------------------------------
# 4. Add synthetic related‑artist interactions
# -------------------------------------------------------------------
def add_related_artist_edges(mat, weight_mat, user_id_to_idx, artist_enc,
                             links_df, mapping, edge_weight=EDGE_WEIGHT):
    artist_to_related = defaultdict(list)
    for _, row in links_df.iterrows():
        aid = row['artist_id']
        rid = row['related_artist_id']
        can_a = mapping.get(aid, aid)
        can_r = mapping.get(rid, rid)
        if can_a in artist_enc.classes_ and can_r in artist_enc.classes_:
            a_idx = artist_enc.transform([can_a])[0]
            r_idx = artist_enc.transform([can_r])[0]
            if a_idx != r_idx:
                artist_to_related[a_idx].append(r_idx)

    mat_coo = mat.tocoo()
    new_rows, new_cols, new_weights = [], [], []
    for u, a in zip(mat_coo.row, mat_coo.col):
        if a in artist_to_related:
            for r in artist_to_related[a]:
                if mat[u, r] == 0:
                    new_rows.append(u)
                    new_cols.append(r)
                    new_weights.append(edge_weight)

    if new_rows:
        n_users, n_artists = mat.shape
        orig_rows = mat_coo.row
        orig_cols = mat_coo.col
        orig_data = mat_coo.data
        orig_weights = weight_mat.data

        all_rows = np.concatenate([orig_rows, new_rows])
        all_cols = np.concatenate([orig_cols, new_cols])
        all_data = np.concatenate([orig_data, np.ones(len(new_rows), dtype=np.float32)])
        all_weights = np.concatenate([orig_weights, new_weights])

        mat_new = csr_matrix((all_data, (all_rows, all_cols)), shape=(n_users, n_artists))
        weight_mat_new = csr_matrix((all_weights, (all_rows, all_cols)), shape=(n_users, n_artists))
        return mat_new, weight_mat_new
    return mat, weight_mat

# -------------------------------------------------------------------
# 5. Build track‑level split (returns train/test matrices and reactors)
# -------------------------------------------------------------------
def build_track_split(df_tr, df_tracks, score_map, test_percent=0.2):
    interactions = []
    for _, row in df_tr.iterrows():
        uid = row['user_id']
        tid = row['track_id']
        reaction_id = row['reaction_id']
        score = score_map.get(reaction_id, 0.0)
        if score > POSITIVE_SCORE_THRESHOLD:
            interactions.append((uid, tid))

    user_enc = LabelEncoder()
    item_enc = LabelEncoder()
    u_idx = user_enc.fit_transform([x[0] for x in interactions])
    i_idx = item_enc.fit_transform([x[1] for x in interactions])

    mat = csr_matrix((np.ones(len(interactions)), (u_idx, i_idx)),
                     shape=(len(user_enc.classes_), len(item_enc.classes_)))

    coo = mat.tocoo()
    np.random.seed(SEED)
    mask = np.random.rand(len(coo.data)) < (1 - test_percent)
    train = csr_matrix((coo.data[mask], (coo.row[mask], coo.col[mask])), shape=mat.shape)
    test  = csr_matrix((coo.data[~mask], (coo.row[~mask], coo.col[~mask])), shape=mat.shape)

    test_user_tracks = defaultdict(set)
    test_coo = test.tocoo()
    for u, i, v in zip(test_coo.row, test_coo.col, test_coo.data):
        if v > 0:
            uid = user_enc.classes_[u]
            tid = item_enc.classes_[i]
            test_user_tracks[uid].add(tid)

    track_pop = np.array(mat.sum(axis=0)).flatten()
    track_id_to_idx = {tid: i for i, tid in enumerate(item_enc.classes_)}

    seen_tracks = defaultdict(set)
    train_coo = train.tocoo()
    for u, i in zip(train_coo.row, train_coo.col):
        uid = user_enc.classes_[u]
        tid = item_enc.classes_[i]
        seen_tracks[uid].add(tid)

    track_reactors = defaultdict(set)
    for u, i in zip(train_coo.row, train_coo.col):
        track_reactors[i].add(u)

    return test_user_tracks, seen_tracks, track_pop, track_id_to_idx, track_reactors, train, test, user_enc, item_enc

# -------------------------------------------------------------------
# 6. Datasets and models for artist MF
# -------------------------------------------------------------------
class ArtistBPRDataset(Dataset):
    def __init__(self, mat, weight_mat, num_neg=1):
        coo = mat.tocoo()
        self.users = torch.LongTensor(coo.row)
        self.pos_items = torch.LongTensor(coo.col)
        self.weights = torch.FloatTensor(weight_mat[coo.row, coo.col].A1)
        self.n_items = mat.shape[1]
        self.num_neg = num_neg

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        user = self.users[idx]
        pos = self.pos_items[idx]
        w = self.weights[idx]
        negs = []
        while len(negs) < self.num_neg:
            neg = random.randint(0, self.n_items - 1)
            if neg != pos:
                negs.append(neg)
        return user, pos, w, torch.LongTensor(negs)

class HybridArtistMF(nn.Module):
    def __init__(self, n_users, n_artists, emb_dim, genre_dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.artist_id_emb = nn.Embedding(n_artists, emb_dim)
        self.genre_proj = nn.Linear(genre_dim, emb_dim, bias=False)
        self.user_bias = nn.Embedding(n_users, 1)
        self.artist_bias = nn.Embedding(n_artists, 1)

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.artist_id_emb.weight)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.artist_bias.weight)

    def forward(self, user_idx, artist_idx, genre_features):
        u = self.user_emb(user_idx)
        a_id = self.artist_id_emb(artist_idx)
        a_genre = self.genre_proj(genre_features[artist_idx])
        a = a_id + a_genre
        dot = (u * a).sum(dim=-1)
        u_bias = self.user_bias(user_idx).squeeze(-1)
        a_bias = self.artist_bias(artist_idx).squeeze(-1)
        return dot + u_bias + a_bias

    def get_artist_embeddings(self, genre_features):
        with torch.no_grad():
            a_id = self.artist_id_emb.weight
            a_genre = self.genre_proj(genre_features)
            return a_id + a_genre

# -------------------------------------------------------------------
# 7. Loss functions for artist model
# -------------------------------------------------------------------
def weighted_bpr_loss(model, user, pos, neg, w, genre_feat):
    pos_score = model(user, pos, genre_feat)
    batch_size, num_neg = neg.shape
    user_exp = user.unsqueeze(1).expand(-1, num_neg).reshape(-1)
    neg_flat = neg.reshape(-1)
    neg_score = model(user_exp, neg_flat, genre_feat).view(batch_size, num_neg)
    diff = pos_score.unsqueeze(1) - neg_score
    bpr = -torch.log(torch.sigmoid(diff) + 1e-10).mean(dim=1)
    return (w * bpr).mean()

def edge_loss(model, edge_a, edge_b, genre_feat):
    a_emb = model.artist_id_emb(edge_a) + model.genre_proj(genre_feat[edge_a])
    b_emb = model.artist_id_emb(edge_b) + model.genre_proj(genre_feat[edge_b])
    return ((a_emb - b_emb) ** 2).mean()

# -------------------------------------------------------------------
# 8. Train hybrid artist model
# -------------------------------------------------------------------
def train_artist_model(mat, weight_mat, genre_feat, artist_enc, edge_indices, epochs):
    n_users, n_artists = mat.shape
    genre_dim = genre_feat.shape[1]
    dataset = ArtistBPRDataset(mat, weight_mat, num_neg=1)
    dataloader = DataLoader(dataset, batch_size=ARTIST_BATCH_SIZE, shuffle=True)

    model = HybridArtistMF(n_users, n_artists, ARTIST_EMBEDDING_DIM, genre_dim).to(DEVICE)
    genre_feat = genre_feat.to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=ARTIST_LR, weight_decay=ARTIST_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    edge_tensor = torch.LongTensor(edge_indices).to(DEVICE)

    for epoch in range(1, epochs+1):
        model.train()
        total_bpr = 0.0
        total_edge = 0.0
        batches = 0
        for user, pos, w, negs in dataloader:
            user, pos, w, negs = user.to(DEVICE), pos.to(DEVICE), w.to(DEVICE), negs.to(DEVICE)

            optimizer.zero_grad()
            bpr = weighted_bpr_loss(model, user, pos, negs, w, genre_feat)

            if edge_tensor.shape[1] > 0:
                num_edges = min(len(user), edge_tensor.shape[1])
                idx = torch.randperm(edge_tensor.shape[1])[:num_edges]
                e_a, e_b = edge_tensor[0, idx], edge_tensor[1, idx]
                e_loss = edge_loss(model, e_a, e_b, genre_feat)
            else:
                e_loss = 0.0

            loss = bpr + REG_LAMBDA * e_loss
            loss.backward()
            optimizer.step()

            total_bpr += bpr.item()
            total_edge += e_loss.item() if isinstance(e_loss, torch.Tensor) else e_loss
            batches += 1

        scheduler.step()
        if epoch % 5 == 0 or epoch == epochs:
            print(f"Artist model epoch {epoch:03d}  BPR={total_bpr/batches:.4f}  Edge={total_edge/batches:.6f}")

    model.eval()
    with torch.no_grad():
        user_emb = model.user_emb.weight.cpu().numpy()
        artist_emb = model.get_artist_embeddings(genre_feat).cpu().numpy()
    return model, user_emb, artist_emb

# -------------------------------------------------------------------
# 9. Train track‑level BPR‑MF model
# -------------------------------------------------------------------
class TrackBPRDataset(Dataset):
    def __init__(self, mat, num_neg=1):
        coo = mat.tocoo()
        self.users = torch.LongTensor(coo.row)
        self.pos_items = torch.LongTensor(coo.col)
        self.n_items = mat.shape[1]
        self.num_neg = num_neg

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        user = self.users[idx]
        pos = self.pos_items[idx]
        negs = []
        while len(negs) < self.num_neg:
            neg = random.randint(0, self.n_items - 1)
            if neg != pos:
                negs.append(neg)
        return user, pos, torch.LongTensor(negs)

class TrackMF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def forward(self, user, item):
        u = self.user_emb(user)
        i = self.item_emb(item)
        return (u * i).sum(dim=-1)

def track_bpr_loss(model, user, pos, neg):
    pos_score = model(user, pos)
    batch_size, num_neg = neg.shape
    user_exp = user.unsqueeze(1).expand(-1, num_neg).reshape(-1)
    neg_flat = neg.reshape(-1)
    neg_score = model(user_exp, neg_flat).view(batch_size, num_neg)
    diff = pos_score.unsqueeze(1) - neg_score
    return -torch.log(torch.sigmoid(diff) + 1e-10).mean()

def train_track_model(train_mat, epochs):
    n_users, n_items = train_mat.shape
    dataset = TrackBPRDataset(train_mat, num_neg=1)
    dataloader = DataLoader(dataset, batch_size=TRACK_BATCH_SIZE, shuffle=True)

    model = TrackMF(n_users, n_items, TRACK_EMBEDDING_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=TRACK_LR, weight_decay=TRACK_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0.0
        batches = 0
        for user, pos, negs in dataloader:
            user, pos, negs = user.to(DEVICE), pos.to(DEVICE), negs.to(DEVICE)
            optimizer.zero_grad()
            loss = track_bpr_loss(model, user, pos, negs)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            batches += 1
        scheduler.step()
        if epoch % 20 == 0 or epoch == epochs:
            print(f"Track model epoch {epoch:03d}  loss={total_loss/batches:.4f}")

    model.eval()
    with torch.no_grad():
        user_emb = model.user_emb.weight.cpu().numpy()
        item_emb = model.item_emb.weight.cpu().numpy()
    return model, user_emb, item_emb

# -------------------------------------------------------------------
# 10. Ensemble evaluation
# -------------------------------------------------------------------
def ensemble_evaluate(artist_user_emb, artist_artist_emb, user_id_to_idx_artist,
                      artist_enc, track_to_artist,
                      track_model, track_user_enc, track_item_enc,
                      test_user_tracks, seen_tracks, train_track_mat,
                      track_pop, track_id_to_idx,   # <-- added these two
                      n_artists=N_ARTISTS_CANDIDATES,
                      tracks_per_artist=TRACKS_PER_ARTIST_CANDIDATE,
                      top_k=TOP_K_FINAL):
    # Map artist index -> list of track ids
    artist_to_tracks = defaultdict(list)
    for tid, artist_id in track_to_artist.items():
        if artist_id in artist_enc.classes_:
            aidx = artist_enc.transform([artist_id])[0]
            artist_to_tracks[aidx].append(tid)

    # For track model, convert user_id to its index
    user_id_to_idx_track = {uid: i for i, uid in enumerate(track_user_enc.classes_)}
    track_item_id_to_idx = {tid: i for i, tid in enumerate(track_item_enc.classes_)}

    # Detach and convert to numpy
    track_user_emb = track_model.user_emb.weight.detach().cpu().numpy()
    track_item_emb = track_model.item_emb.weight.detach().cpu().numpy()

    recalls, precisions = [], []
    for uid, test_tids in test_user_tracks.items():
        if uid not in user_id_to_idx_artist:
            continue
        u_artist = user_id_to_idx_artist[uid]
        # 1. Get top artists from artist model
        scores_a = artist_user_emb[u_artist].dot(artist_artist_emb.T)
        top_artists = np.argpartition(scores_a, -n_artists)[-n_artists:]
        top_artists = top_artists[np.argsort(scores_a[top_artists])[::-1]]

        seen = seen_tracks.get(uid, set())
        # Collect candidate tracks from top artists
        candidate_tids = set()
        for aidx in top_artists:
            track_list = artist_to_tracks.get(aidx, [])
            for t in track_list:
                if t not in seen:
                    candidate_tids.add(t)

        if len(candidate_tids) == 0:
            recalls.append(0.0)
            precisions.append(0.0)
            continue

        # 2. Re-rank candidates with track model
        if uid in user_id_to_idx_track:
            u_track = user_id_to_idx_track[uid]
            u_vec = track_user_emb[u_track]
            scores = []
            for t in candidate_tids:
                if t in track_item_id_to_idx:
                    i_idx = track_item_id_to_idx[t]
                    s = np.dot(u_vec, track_item_emb[i_idx])
                    scores.append((t, s))
            scores.sort(key=lambda x: x[1], reverse=True)
            top_tracks = [t for t,_ in scores[:top_k]]
        else:
            # fallback: use global popularity
            top_tracks = sorted(candidate_tids,
                                key=lambda t: track_pop[track_id_to_idx.get(t, -1)] if t in track_id_to_idx else 0,
                                reverse=True)[:top_k]

        hit = len(set(top_tracks) & test_tids)
        recall = hit / len(test_tids) if test_tids else 0.0
        precision = hit / top_k
        recalls.append(recall)
        precisions.append(precision)

    return np.mean(recalls) if recalls else 0.0, np.mean(precisions) if precisions else 0.0

# -------------------------------------------------------------------
# 11. Main
# -------------------------------------------------------------------
def main():
    # Load data
    df_tr, df_tracks, df_artists, df_ar, df_rt = load_data(
        'processed/track_reactions.csv',
        'processed/tracks.csv',
        'processed/artists_1.csv',
        'processed/artist_reactions.csv',
        'processed/reaction_types.csv'
    )
    score_map = build_score_map(df_rt)
    mapping = load_canonical_mapping('artist_dedup_mapping.csv')

    # =======================  Artist model  =======================
    print("Building weighted user‑artist matrix...")
    ua_mat, weight_mat, user_id_to_idx_artist, artist_enc, track_to_artist, user_enc_artist = build_user_artist_matrix(
        df_tr, df_tracks, df_ar, score_map, mapping
    )
    print(f"User‑Artist matrix: {ua_mat.shape[0]} users x {ua_mat.shape[1]} artists, {ua_mat.nnz} interactions")

    if USE_SYNTHETIC_EDGES:
        links_df = pd.read_csv('artist_links.csv')
        print("Adding synthetic related‑artist interactions...")
        ua_mat, weight_mat = add_related_artist_edges(
            ua_mat, weight_mat, user_id_to_idx_artist, artist_enc, links_df, mapping
        )
        print(f"After augmentation: {ua_mat.nnz} interactions")

    # Track split (needed for evaluation)
    (test_user_tracks, seen_tracks, track_pop, track_id_to_idx,
     track_reactors, train_track_mat, test_track_mat,
     track_user_enc, track_item_enc) = build_track_split(df_tr, df_tracks, score_map, TEST_PERCENT)

    # Genre features (aligned)
    genre_sparse = load_npz('artist_genre_matrix_final.npz')
    genre_feat_full = torch.FloatTensor(genre_sparse.toarray())
    canonical_ids = pd.read_csv('canonical_artist_ids.csv')['canonical_artist_id'].tolist()
    canonical_to_row = {cid: i for i, cid in enumerate(canonical_ids)}
    selected_rows = [canonical_to_row[aid] for aid in artist_enc.classes_]
    genre_feat = genre_feat_full[selected_rows]

    # Related‑artist edges for regularisation loss
    edge_indices = []
    for _, row in links_df.iterrows():
        aid = row['artist_id']
        rid = row['related_artist_id']
        can_a = mapping.get(aid, aid)
        can_r = mapping.get(rid, rid)
        if can_a in artist_enc.classes_ and can_r in artist_enc.classes_:
            a_idx = artist_enc.transform([can_a])[0]
            r_idx = artist_enc.transform([can_r])[0]
            if a_idx != r_idx:
                edge_indices.append([a_idx, r_idx])
    edge_indices = np.array(edge_indices).T

    print("Training hybrid artist model...")
    artist_model, artist_user_emb, artist_artist_emb = train_artist_model(
        ua_mat, weight_mat, genre_feat, artist_enc, edge_indices, ARTIST_EPOCHS
    )

    # =======================  Track model  =======================
    print("Training track‑level BPR‑MF model...")
    track_model, track_user_emb, track_item_emb = train_track_model(train_track_mat, TRACK_EPOCHS)

    # =======================  Ensemble evaluation  =======================
    if USE_ENSEMBLE:
        print("Ensemble evaluation (artist + track re‑ranking)...")
        rec, prec = ensemble_evaluate(
            artist_user_emb, artist_artist_emb, user_id_to_idx_artist,
            artist_enc, track_to_artist,
            track_model, track_user_enc, track_item_enc,
            test_user_tracks, seen_tracks, train_track_mat,
            track_pop, track_id_to_idx          # added
        )
        print(f"Track Recall@10: {rec:.4f}   Precision@10: {prec:.4f}")
    else:
        # Fall back to artist‑only evaluation with personalised scoring (not shown here)
        pass

    # =======================  Export  =======================
    os.makedirs(EXPORT_DIR, exist_ok=True)
    torch.save(artist_model.state_dict(), os.path.join(EXPORT_DIR, 'artist_model_state.pt'))
    torch.save(track_model.state_dict(), os.path.join(EXPORT_DIR, 'track_model_state.pt'))
    with open(os.path.join(EXPORT_DIR, 'user_enc_artist.pkl'), 'wb') as f:
        pickle.dump(user_enc_artist, f)
    with open(os.path.join(EXPORT_DIR, 'artist_enc.pkl'), 'wb') as f:
        pickle.dump(artist_enc, f)
    with open(os.path.join(EXPORT_DIR, 'track_user_enc.pkl'), 'wb') as f:
        pickle.dump(track_user_enc, f)
    with open(os.path.join(EXPORT_DIR, 'track_item_enc.pkl'), 'wb') as f:
        pickle.dump(track_item_enc, f)
    with open(os.path.join(EXPORT_DIR, 'track_to_artist.pkl'), 'wb') as f:
        pickle.dump(track_to_artist, f)
    with open(os.path.join(EXPORT_DIR, 'track_id_to_idx.pkl'), 'wb') as f:
        pickle.dump(track_id_to_idx, f)
    np.save(os.path.join(EXPORT_DIR, 'track_pop.npy'), track_pop)
    np.save(os.path.join(EXPORT_DIR, 'artist_user_embeddings.npy'), artist_user_emb)
    np.save(os.path.join(EXPORT_DIR, 'artist_artist_embeddings.npy'), artist_artist_emb)
    np.save(os.path.join(EXPORT_DIR, 'track_user_embeddings.npy'), track_user_emb)
    np.save(os.path.join(EXPORT_DIR, 'track_item_embeddings.npy'), track_item_emb)
    torch.save(genre_feat, os.path.join(EXPORT_DIR, 'genre_features.pt'))

    config = {
        'artist_embedding_dim': ARTIST_EMBEDDING_DIM,
        'track_embedding_dim': TRACK_EMBEDDING_DIM,
        'n_users_artist': len(user_enc_artist.classes_),
        'n_artists': len(artist_enc.classes_),
        'genre_dim': genre_feat.shape[1],
        'n_artists_candidates': N_ARTISTS_CANDIDATES,
        'tracks_per_artist_candidate': TRACKS_PER_ARTIST_CANDIDATE,
        'top_k_final': TOP_K_FINAL
    }
    with open(os.path.join(EXPORT_DIR, 'ensemble_config.json'), 'w') as f:
        json.dump(config, f, indent=2)

    print(f"All models and artifacts saved to {EXPORT_DIR}")

if __name__ == "__main__":
    main()

Building weighted user‑artist matrix...
User‑Artist matrix: 1285 users x 4919 artists, 59959 interactions
Adding synthetic related‑artist interactions...
After augmentation: 145685 interactions
Training hybrid artist model...
Artist model epoch 005  BPR=0.8007  Edge=0.047265
Artist model epoch 010  BPR=0.5602  Edge=0.064830
Artist model epoch 015  BPR=0.4931  Edge=0.073584
Artist model epoch 020  BPR=0.4661  Edge=0.077632
Artist model epoch 025  BPR=0.4369  Edge=0.079042
Artist model epoch 030  BPR=0.4121  Edge=0.079831
Artist model epoch 035  BPR=0.4027  Edge=0.079100
Artist model epoch 040  BPR=0.3977  Edge=0.078784
Artist model epoch 045  BPR=0.3678  Edge=0.077522
Artist model epoch 050  BPR=0.3625  Edge=0.075651
Artist model epoch 055  BPR=0.3491  Edge=0.074002
Artist model epoch 060  BPR=0.3309  Edge=0.072297
Artist model epoch 065  BPR=0.3257  Edge=0.070812
Artist model epoch 070  BPR=0.3123  Edge=0.069564
Artist model epoch 075  BPR=0.3104  Edge=0.068279
Artist model epoch 080  

In [12]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pickle
import json
import random
from collections import defaultdict
from scipy.sparse import csr_matrix

# -------------------------------------------------------------------
# Configuration – adjust paths as needed
# -------------------------------------------------------------------
MODEL_DIR = 'model_params_colab'
DATA_DIR = 'processed'   # folder containing the original CSV files

# -------------------------------------------------------------------
# Model classes (must match training code exactly)
# -------------------------------------------------------------------
class HybridArtistMF(nn.Module):
    def __init__(self, n_users, n_artists, emb_dim, genre_dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.artist_id_emb = nn.Embedding(n_artists, emb_dim)
        self.genre_proj = nn.Linear(genre_dim, emb_dim, bias=False)
        self.user_bias = nn.Embedding(n_users, 1)
        self.artist_bias = nn.Embedding(n_artists, 1)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.artist_id_emb.weight)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.artist_bias.weight)

    def forward(self, user_idx, artist_idx, genre_features):
        u = self.user_emb(user_idx)
        a_id = self.artist_id_emb(artist_idx)
        a_genre = self.genre_proj(genre_features[artist_idx])
        a = a_id + a_genre
        dot = (u * a).sum(dim=-1)
        u_bias = self.user_bias(user_idx).squeeze(-1)
        a_bias = self.artist_bias(artist_idx).squeeze(-1)
        return dot + u_bias + a_bias

class TrackMF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def forward(self, user, item):
        u = self.user_emb(user)
        i = self.item_emb(item)
        return (u * i).sum(dim=-1)

# -------------------------------------------------------------------
# Load model artifacts
# -------------------------------------------------------------------
print("Loading models and artifacts...")
with open(f'{MODEL_DIR}/ensemble_config.json', 'r') as f:
    config = json.load(f)

with open(f'{MODEL_DIR}/user_enc_artist.pkl', 'rb') as f:
    user_enc_artist = pickle.load(f)
with open(f'{MODEL_DIR}/artist_enc.pkl', 'rb') as f:
    artist_enc = pickle.load(f)
with open(f'{MODEL_DIR}/track_user_enc.pkl', 'rb') as f:
    track_user_enc = pickle.load(f)
with open(f'{MODEL_DIR}/track_item_enc.pkl', 'rb') as f:
    track_item_enc = pickle.load(f)
with open(f'{MODEL_DIR}/track_to_artist.pkl', 'rb') as f:
    track_to_artist = pickle.load(f)
with open(f'{MODEL_DIR}/track_id_to_idx.pkl', 'rb') as f:
    track_id_to_idx = pickle.load(f)

track_pop = np.load(f'{MODEL_DIR}/track_pop.npy')
genre_feat = torch.load(f'{MODEL_DIR}/genre_features.pt', map_location='cpu')

# Instantiate models and load state dicts
artist_model = HybridArtistMF(
    config['n_users_artist'], config['n_artists'],
    config['artist_embedding_dim'], config['genre_dim']
)
artist_model.load_state_dict(torch.load(f'{MODEL_DIR}/artist_model_state.pt', map_location='cpu'))
artist_model.eval()

track_model = TrackMF(
    len(track_user_enc.classes_), len(track_item_enc.classes_),
    config['track_embedding_dim']
)
track_model.load_state_dict(torch.load(f'{MODEL_DIR}/track_model_state.pt', map_location='cpu'))
track_model.eval()

# Compute artist embeddings (includes genre projection)
with torch.no_grad():
    artist_embeddings = artist_model.artist_id_emb.weight + artist_model.genre_proj(genre_feat)
    artist_embeddings = artist_embeddings.numpy()
    artist_user_embeddings = artist_model.user_emb.weight.numpy()
    track_user_embeddings = track_model.user_emb.weight.detach().numpy()
    track_item_embeddings = track_model.item_emb.weight.detach().numpy()

# -------------------------------------------------------------------
# Load original data for display
# -------------------------------------------------------------------
# -------------------------------------------------------------------
# Load original data for display
# -------------------------------------------------------------------
tracks_df = pd.read_csv(f'{DATA_DIR}/tracks.csv')   # columns: id, artists_id, ...
# Build track_id → list of artist names (can be multiple, we'll take the first)
# We'll map artist_id to artist name later
artists_df = pd.read_csv(f'{DATA_DIR}/artists_1.csv')
artist_id_to_name = dict(zip(artists_df['id'], artists_df['name']))

# Load track reactions to get user history
tr_df = pd.read_csv(f'{DATA_DIR}/track_reactions.csv')
rt_df = pd.read_csv(f'{DATA_DIR}/reaction_types.csv')
score_map = dict(zip(rt_df['id'], rt_df['score']))

# Build user positive interactions (for display only)
user_liked_tracks = defaultdict(list)
for _, row in tr_df.iterrows():
    reaction_id = row['reaction_id']
    score = score_map.get(reaction_id, 0.0)
    if score > 1.0:   # positive threshold (same as training)
        user_liked_tracks[row['user_id']].append(row['track_id'])

# -------------------------------------------------------------------
# Select 3 random users with at least 5 liked tracks
# -------------------------------------------------------------------
eligible_users = [uid for uid, tids in user_liked_tracks.items()
                  if uid in user_enc_artist.classes_ and len(tids) >= 5]
random.seed(12598)
selected_users = random.sample(eligible_users, 10)

# -------------------------------------------------------------------
# Recommendation function (ensemble)
# -------------------------------------------------------------------
def recommend_for_user(user_id, n_artists=15, tracks_per_artist=10, top_k=10):
    if user_id not in user_enc_artist.classes_:
        return []
    u_idx_artist = user_enc_artist.transform([user_id])[0]
    user_vec_artist = artist_user_embeddings[u_idx_artist]

    # 1. Artist ranking
    artist_scores = user_vec_artist.dot(artist_embeddings.T)
    top_artists = np.argpartition(artist_scores, -n_artists)[-n_artists:]
    top_artists = top_artists[np.argsort(artist_scores[top_artists])[::-1]]

    # 2. Collect candidate tracks from these artists
    seen = set(user_liked_tracks.get(user_id, []))
    candidates = set()
    for aidx in top_artists:
        artist_id = artist_enc.inverse_transform([aidx])[0]
        for tid, aid in track_to_artist.items():
            if aid == artist_id and tid not in seen:
                candidates.add(tid)

    if not candidates:
        return []

    # 3. Re-rank candidates with track model
    if user_id in track_user_enc.classes_:
        u_idx_track = track_user_enc.transform([user_id])[0]
        u_vec_track = track_user_embeddings[u_idx_track]
        scored = []
        for tid in candidates:
            if tid in track_item_enc.classes_:
                i_idx = track_item_enc.transform([tid])[0]
                s = np.dot(u_vec_track, track_item_embeddings[i_idx])
                scored.append((tid, s))
        scored.sort(key=lambda x: x[1], reverse=True)
        top_tracks = [t for t,_ in scored[:top_k]]
    else:
        top_tracks = sorted(candidates,
                            key=lambda t: track_pop[track_id_to_idx.get(t, -1)] if t in track_id_to_idx else 0,
                            reverse=True)[:top_k]
    return top_tracks

# -------------------------------------------------------------------
# Generate and display recommendations for each selected user
# -------------------------------------------------------------------
print("=" * 70)
for uid in selected_users:
    print(f"\nUser ID: {uid}")

    # Previously liked tracks (sample up to 10)
    liked = user_liked_tracks[uid]
    print(f"Previously liked {len(liked)} tracks (showing up to 10):")
    for tid in liked[:20]:
        # Get artist for this track
        artist_id = track_to_artist.get(tid, None)
        if artist_id:
            artist_name = artist_id_to_name.get(artist_id, 'Unknown Artist')
        else:
            artist_name = 'Unknown Artist'
        print(f"  Track {tid}: by {artist_name}")

    # Get recommendations
    recs = recommend_for_user(uid)
    print(f"\nRecommended tracks:")
    if not recs:
        print("  No recommendations available.")
    else:
        for tid in recs:
            artist_id = track_to_artist.get(tid, None)
            if artist_id:
                artist_name = artist_id_to_name.get(artist_id, 'Unknown Artist')
            else:
                artist_name = 'Unknown Artist'
            print(f"  Track {tid}: by {artist_name}")
    print("-" * 70)

Loading models and artifacts...


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



User ID: 655438819868.0
Previously liked 22 tracks (showing up to 10):
  Track 1012.0: by Pixies
  Track 6968.0: by Rival Sons
  Track 7095.0: by Arctic Monkeys
  Track 7393.0: by ThxSoMch
  Track 7432.0: by The Weeknd
  Track 7433.0: by Enterprise Earth
  Track 1148.0: by The Weeknd
  Track 7436.0: by HIM
  Track 1215.0: by Sufjan Stevens
  Track 1846.0: by Laufey
  Track 7642.0: by Mitski
  Track 1035.0: by Arctic Monkeys
  Track 7649.0: by Dariush
  Track 7650.0: by Sting
  Track 7672.0: by او و دوستانش he and his friends
  Track 7682.0: by Hozier
  Track 7683.0: by Hozier
  Track 7684.0: by Raye
  Track 61.0: by Hozier
  Track 7704.0: by ‌ㅤㅤanathema

Recommended tracks:
  Track 535: by Evanescence
  Track 5789: by Arctic Monkeys
  Track 6526: by Scorpions
  Track 7097: by Arctic Monkeys
  Track 6556: by Scorpions
  Track 4376: by The Weeknd
  Track 8443: by Sufjan Stevens
  Track 2102: by d4vd
  Track 6760: by Arctic Monkeys
  Track 4331: by The Weeknd
----------------------------